## Step 1: The Real-World Problem

### Two business owners

| | **Sita (Bihar)** | **Ramesh (Maharashtra)** |
|---|---|---|
| Business | Small workshop making cardboard boxes by hand | Large factory making auto parts with 20 CNC machines |
| Output today | 100 boxes per day | Already at full capacity |
| Bank credit | None, and no savings | Has a ₹5 crore bank loan |
| What she or he wants | A ₹5 lakh automatic machine (1,000 boxes per day) | A few more spare parts |
| Credit situation | **Credit-starved** | **Credit-saturated** |

### The economic reality

- **Sita:** a ₹5 lakh loan lets her buy the machine, and her output rises **10 times**. Every rupee of credit is used to produce more.
- **Ramesh:** an extra ₹5 lakh barely changes his output. He already has the machines he needs.

This is the idea of **diminishing returns to capital**. The first machine adds a lot of output, and each additional machine adds less.

### From two firms to a whole country

India has many "Sitas" and a few "Ramesh"es, and they are not spread evenly across states.

- **Credit-scarce states** (such as Bihar, Jharkhand and Assam) have smaller loans, more rural branches, and more small, informal firms.
- **Credit-rich states** (such as Maharashtra, Delhi and Tamil Nadu) have large loans, big corporate borrowers, and deep banking.

**Research problem:** if banks lend more to manufacturing, does it raise industrial output, and does the same rupee of credit do more good in credit-scarce states than in credit-rich ones?

**Data used:** a panel of about 30 Indian states, FY 2009-10 to FY 2025-26, built from RBI district-level bank credit (BSR-1) and state income accounts (`master_dataset.csv`).

## Step 2: Research Questions and Hypotheses

### Question 1 (H1): Does bank credit to manufacturing raise industrial output?

**Idea:** more credit lets firms buy more machines, and more machines produce more output.

$$\text{More Credit} \rightarrow \text{More Machines } (K) \rightarrow \text{More Output } (Y)$$

| | Statement |
|---|---|
| $H_0$ | Manufacturing credit has **no effect** on manufacturing output: $\beta_1 = 0$ |
| $H_1$ | Manufacturing credit **raises** manufacturing output: $\beta_1 > 0$ |

**How we measure it**

| Concept | Column in `master_dataset.csv` | Meaning |
|---|---|---|
| Output ($Y$) | `ln_nsdp_manuf_cnst` | Log of manufacturing value added, at constant 2011-12 prices |
| Credit | `l1_ln_real_cr_out_manuf` | Log of last year's real bank credit to manufacturing |

$\beta_1$ is an **elasticity**. For example, $\beta_1 = 0.20$ means a 10% rise in credit is linked to a 2% rise in output.

---

### Question 2 (H2): Where does ₹1 of credit do the most good?

**Idea:** because of diminishing returns, credit should raise output **more** in credit-scarce states (Sita) than in credit-rich states (Ramesh).

```
 Industrial
 Output (Y)
    ^
    |                         /   <-- Credit-scarce state (steep: big gain per rupee)
    |                        /
    |                       /
    |              ________/      <-- Credit-rich state (flat: small gain per rupee)
    |             /
    +--------------------------------------------> Bank Credit
```

| | Statement |
|---|---|
| $H_0$ | The effect of credit is the **same** in all states |
| $H_2$ | The effect of credit is **larger** in states that started with low credit depth |

**How we measure "credit-scarce"**

| Column | Meaning |
|---|---|
| `initial_depth` | Bank credit ÷ GSDP in the first year of the data, so it shows how much banking each state started with |
| `initial_depth_tercile` | States split into 3 groups: **1 = credit-scarce**, 2 = middle, 3 = credit-rich |

We test H2 in two ways:

1. **Interaction:** add $\ln(\text{Credit}) \times \text{InitialDepth}$ to the model. **H2 is supported if its coefficient $\beta_3 < 0$**, meaning credit helps less where banking was already deep.
2. **Split sample:** estimate $\beta_1$ separately for tercile 1 and tercile 3. **H2 is supported if $\beta_1^{\text{scarce}} > \beta_1^{\text{rich}}$**.

---

### A note on "MSME credit"

The RBI data does **not** split loans by firm size (micro, small, medium, large). We therefore use **credit to manufacturing** as the main credit variable, and test the small-firm story with these proxies:

| Proxy | Column | Why it reflects small firms |
|---|---|---|
| Small-ticket share | `small_ticket_share_manuf` | Share of factory credit in loans averaging ≤ ₹25 lakh per account |
| Grassroots share | `grassroots_share_manuf` | Factory credit from RRBs and small finance banks, or from rural and semi-urban branches |
| Average loan size | `avg_ticket_manuf_lakh` | Smaller average loans suggest smaller borrowers |
| Informal output | `ln_nsdp_manuf_unreg_cnst` | Output of unregistered (mostly small) factories, available 2009-10 to 2014-15 |

In [ ]:
# =====================================================================
# STEP 3: Prepare the analysis sample from master_dataset.csv
# =====================================================================
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# ---------------- settings -----------------------------------------
MASTER_PATH = "master_dataset.csv"
OUT_PATH    = "analysis_panel.csv"
YEAR_MIN, YEAR_MAX = 2010, 2025     # 2025-26 dropped: constant-price data/deflator not yet available
MIN_YEARS   = 8                     # a state needs >= 8 complete years to stay in the sample
CORE = ["ln_nsdp_manuf_cnst", "ln_real_cr_out_manuf"]   # outcome + credit must both exist

# ---------------- 3.1 Load and basic checks ------------------------
df = pd.read_csv(MASTER_PATH)
print(f"Loaded {MASTER_PATH}: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"States/UTs: {df['state'].nunique()}  |  Years: {df['year'].min()}-{df['year'].max()}")

dups = df.duplicated(["state", "year"]).sum()
print(f"Duplicate state-year rows: {dups}")
assert dups == 0, "Duplicate state-year rows found - check the merge step."

# ---------------- 3.2 Coverage: which years does each state have? ---
cov_cols = {
    "credit":          "cr_out_manuf",
    "output":          "nsdp_manuf_cnst",
    "informal_output": "nsdp_manuf_unreg_cnst",
    "deflator":        "deflator",
    "gsdp":            "gsdp_curr",
    "nsva":            "nsva_total_curr",
    "cpi":             "cpi_fy",
}
def span(s, years):
    y = years[s.notna()]
    return f"{y.min()}-{y.max()} ({len(y)})" if len(y) else "-"

coverage = pd.DataFrame({
    name: df.groupby("state").apply(lambda g, c=col: span(g[c], g["year"]), include_groups=False)
    for name, col in cov_cols.items()
})
print("\n3.2 Coverage by state: first-last year (number of years)")
display(coverage)

# ---------------- 3.3 Missing values in key columns ----------------
key = ["cr_out_manuf", "real_cr_out_manuf", "nsdp_manuf_cnst", "nsdp_manuf_unreg_cnst",
       "deflator", "pc_nsdp_cnst", "credit_depth_gsdp", "credit_depth_nsva",
       "initial_depth", "small_ticket_share_manuf", "grassroots_share_manuf"]
miss = (df[key].isna().mean() * 100).round(1).rename("% missing").to_frame()
print("\n3.3 Missing values (all rows, before filtering)")
display(miss)

# ---------------- 3.4 Sanity checks --------------------------------
print("\n3.4 Sanity checks")
amounts = [c for c in df.columns if c.startswith(("cr_out", "cr_lim", "cr_acc", "nsdp_", "nsva_", "gsdp_", "pc_nsdp"))]
neg = (df[amounts] < 0).sum()
print("  Negative values in amount columns:", int(neg.sum()), "" if neg.sum() == 0 else neg[neg > 0].to_dict())

shares = [c for c in df.columns if "share" in c]
bad_share = ((df[shares] < 0) | (df[shares] > 1)).sum()
print("  Shares outside 0-1:", int(bad_share.sum()), "" if bad_share.sum() == 0 else bad_share[bad_share > 0].to_dict())

over = df.loc[df["util_manuf"] > 1, ["state", "year", "util_manuf"]]
print(f"  Manufacturing utilisation > 1 (borrowed more than the limit): {len(over)} rows")

jumps = df.loc[df["g_real_cr_out_manuf"].abs() > 0.7, ["state", "year", "g_real_cr_out_manuf"]]
print(f"  Extreme credit jumps (|growth| > 70% in one year): {len(jumps)} rows (largest 15 shown)")
if len(jumps):
    display(jumps.reindex(jumps["g_real_cr_out_manuf"].abs().sort_values(ascending=False).index).head(15))

jumps_y = df.loc[df["g_nsdp_manuf_cnst"].abs() > 0.5, ["state", "year", "g_nsdp_manuf_cnst"]]
print(f"  Extreme output jumps (|growth| > 50% in one year): {len(jumps_y)} rows (largest 15 shown)")
if len(jumps_y):
    display(jumps_y.reindex(jumps_y["g_nsdp_manuf_cnst"].abs().sort_values(ascending=False).index).head(15))

# ---------------- 3.5 Define the analysis sample -------------------
p = df[df["year"].between(YEAR_MIN, YEAR_MAX)].copy()
p["complete"] = p[CORE].notna().all(axis=1)
n_complete = p.groupby("state")["complete"].sum()
keep = n_complete[n_complete >= MIN_YEARS].index
dropped = n_complete[n_complete < MIN_YEARS].sort_values()

print(f"\n3.5 Sample rule: years {YEAR_MIN}-{YEAR_MAX}, both {CORE} present, at least {MIN_YEARS} such years")
print(f"  Kept {len(keep)} states; dropped {len(dropped)}:")
display(dropped.rename("complete years").to_frame())

panel = p[p["state"].isin(keep) & p["complete"]].drop(columns="complete").copy()
if panel.empty:
    raise ValueError("No state meets the sample rule. Check that master_dataset.csv was built from the "
                     "FULL files (not sample.csv), or lower MIN_YEARS.")


# ---------------- 3.5b Andhra Pradesh / Telangana split --------------
# Before FY2013-14, AP's credit includes Hyderabad/Telangana -> not the same territory.
AP, AP_SPLIT_YEAR = "Andhra Pradesh", 2014
if AP in panel["state"].values:
    n0 = len(panel)
    panel = panel[~((panel["state"] == AP) & (panel["year"] < AP_SPLIT_YEAR))].copy()
    ap_rows = panel["state"] == AP
    # AP's first remaining year must not use undivided values as its lag / growth base
    first_ap = ap_rows & (panel["year"] == panel.loc[ap_rows, "year"].min())
    lag_cols = [c for c in panel.columns if c.startswith(("l1_", "g_"))]
    panel.loc[first_ap, lag_cols] = np.nan
    # AP's initial credit depth from its first post-split year (not undivided 2009-10)
    old = panel.loc[ap_rows, "initial_depth"].iloc[0]
    for src in ["credit_depth_gsdp", "credit_depth_nsva"]:
        ok = panel.loc[ap_rows].dropna(subset=[src]).sort_values("year")
        if len(ok):
            panel.loc[ap_rows, "initial_depth"] = ok[src].iloc[0]
            break
    print(f"\n3.5b Andhra Pradesh: dropped {n0 - len(panel)} pre-split rows; initial depth "
          f"{old:.3f} (undivided) -> {panel.loc[ap_rows, 'initial_depth'].iloc[0]:.3f} "
          f"({src}, first post-split year)")

# ---------------- 3.6 Extra variables for later steps --------------
# Terciles re-computed WITHIN the analysis sample (UTs no longer distort the cut-offs)
init = panel.groupby("state")["initial_depth"].first().dropna()
if len(init) >= 3:
    panel["depth_tercile"] = panel["state"].map(
        pd.qcut(init.rank(method="first"), 3, labels=[1, 2, 3]).astype(int))
else:
    panel["depth_tercile"] = np.nan
panel["low_depth"] = (panel["depth_tercile"] == 1).astype(int)       # credit-scarce dummy
panel["covid"]     = panel["year"].isin([2021, 2022]).astype(int)   # FY 2020-21, 2021-22
panel["trend"]     = panel["year"] - panel["year"].min()

# Lagged control (income per person, last year)
panel = panel.sort_values(["state", "year"])
lag_ok = panel.groupby("state")["year"].diff() == 1
panel["l1_ln_pc_nsdp_cnst"] = panel.groupby("state")["ln_pc_nsdp_cnst"].shift(1).where(lag_ok)

# ---------------- 3.7 Panel structure summary ----------------------
T = panel.groupby("state").size()
print("\n3.7 Final analysis panel")
print(f"  Observations: {len(panel)}  |  States (N): {panel['state'].nunique()}  |  "
      f"Years: {panel['year'].min()}-{panel['year'].max()}")
print(f"  Years per state: min {T.min()}, mean {T.mean():.1f}, max {T.max()}  ->  "
      f"{'balanced' if T.nunique() == 1 else 'unbalanced'} panel")
print(f"  Rows usable for the lagged-credit model: {panel['l1_ln_real_cr_out_manuf'].notna().sum()}")
print(f"  Rows with informal-manufacturing output: {panel['ln_nsdp_manuf_unreg_cnst'].notna().sum()}")

print("\n  States by initial credit depth (1 = credit-scarce, 3 = credit-rich):")
terc = (panel.groupby("state")[["initial_depth", "depth_tercile"]].first()
        .sort_values("initial_depth").round(3))
display(terc)

# ---------------- 3.8 Save -----------------------------------------
panel.to_csv(OUT_PATH, index=False)
print(f"\nSaved {OUT_PATH}  ({panel.shape[0]} rows x {panel.shape[1]} columns)")

In [ ]:
# =====================================================================
# STEP 3b: Diagnose extreme output jumps + add robustness variables
# Run AFTER Step 3 (needs master_dataset.csv and analysis_panel.csv)
# =====================================================================
import numpy as np
import pandas as pd
from IPython.display import display

master = pd.read_csv("master_dataset.csv")
panel  = pd.read_csv("analysis_panel.csv")

# ---------- 1. Look at each jump: is it in current prices too, or only constant? ----------
JUMP = 0.5
jumps = master.loc[master["g_nsdp_manuf_cnst"].abs() > JUMP, ["state", "year"]]

master = master.sort_values(["state", "year"])
master["g_curr"] = master.groupby("state")["nsdp_manuf_curr"].transform(lambda s: np.log(s).diff())
master["g_defl"] = master.groupby("state")["deflator"].transform(lambda s: np.log(s).diff())

rows = []
for _, j in jumps.iterrows():
    w = master[(master["state"] == j["state"]) & master["year"].between(j["year"] - 2, j["year"] + 1)]
    rows.append(w.assign(jump_year=j["year"]))
detail = pd.concat(rows)[["state", "jump_year", "year", "nsdp_manuf_curr", "nsdp_manuf_cnst",
                          "g_nsdp_manuf_cnst", "g_curr", "g_defl", "manuf_share", "cr_out_manuf"]]
print("Output around each jump year (Rs crore; g_ = log growth vs previous year)")
display(detail.round(3))

summary = master.loc[jumps.index, ["state", "year", "g_nsdp_manuf_cnst", "g_curr", "g_defl", "nsdp_manuf_cnst"]]
summary["same_in_current_prices"] = (np.sign(summary["g_curr"]) == np.sign(summary["g_nsdp_manuf_cnst"])) & \
                                    (summary["g_curr"].abs() > JUMP * 0.7)
summary["size_rank"] = master.groupby("year")["nsdp_manuf_cnst"].rank(ascending=False).loc[jumps.index]
print("\nSummary: a jump that also shows in current prices is in the source data, not caused by deflating")
display(summary.round(3).sort_values("g_nsdp_manuf_cnst"))

# ---------- 2. Robustness variables in the analysis panel ----------
# (a) small manufacturing base: bottom quartile of average real manufacturing output
avg_out = panel.groupby("state")["nsdp_manuf_cnst"].mean()
small = avg_out[avg_out <= avg_out.quantile(0.25)].index
panel["small_manuf_state"] = panel["state"].isin(small).astype(int)

# (b) jump flag on the row where the jump happens
panel["output_jump"] = (panel["g_nsdp_manuf_cnst"].abs() > JUMP).astype(int)

# (c) winsorised growth rates (cap at 1st / 99th percentile) for growth-rate models
def winsor(s, lo=0.01, hi=0.99):
    return s.clip(s.quantile(lo), s.quantile(hi))
panel["g_nsdp_manuf_cnst_w"]   = winsor(panel["g_nsdp_manuf_cnst"])
panel["g_real_cr_out_manuf_w"] = winsor(panel["g_real_cr_out_manuf"])

print(f"\nSmall-manufacturing states (bottom 25% by average output): {sorted(small)}")
print(f"Rows flagged output_jump = 1 in the analysis panel: {panel['output_jump'].sum()}")
print("Growth after winsorising (min / max):",
      round(panel["g_nsdp_manuf_cnst_w"].min(), 3), "/", round(panel["g_nsdp_manuf_cnst_w"].max(), 3))

panel.to_csv("analysis_panel.csv", index=False)
print("\nUpdated analysis_panel.csv with: small_manuf_state, output_jump, g_nsdp_manuf_cnst_w, g_real_cr_out_manuf_w")

In [ ]:
# =====================================================================
# STEP 3c: Flag one-year "blips" and add an alternative output measure
# Run AFTER Step 3b (updates analysis_panel.csv)
# =====================================================================
import numpy as np
import pandas as pd
from IPython.display import display

panel = pd.read_csv("analysis_panel.csv").sort_values(["state", "year"])
g = panel.groupby("state")
consec_next = g["year"].shift(-1) - panel["year"] == 1

# 1) Blip = output jumps one way by >30% and comes straight back the next year (V or inverted-V)
g_now  = panel["g_nsdp_manuf_cnst"]
g_next = g["g_nsdp_manuf_cnst"].shift(-1).where(consec_next)
panel["output_blip"] = ((g_now.abs() > 0.3) & (g_next.abs() > 0.3)
                        & (np.sign(g_now) != np.sign(g_next))).astype(int)

# 2) Alternative real output: current-price manufacturing / state price deflator
#    (same deflator as credit; avoids breaks in the source constant-price series, e.g. Puducherry 2019)
panel["ln_nsdp_manuf_defl"] = np.log((panel["nsdp_manuf_curr"] / panel["deflator"]).where(lambda s: s > 0))
lag_ok = g["year"].diff() == 1
panel["g_nsdp_manuf_defl"] = panel["ln_nsdp_manuf_defl"] - g["ln_nsdp_manuf_defl"].shift(1).where(lag_ok)

# 3) One flag for "drop these rows in the robustness check"
panel["suspect_obs"] = ((panel["output_blip"] == 1) | (panel["output_jump"] == 1)).astype(int)

print("Rows flagged as one-year blips (the odd year itself):")
display(panel.loc[panel["output_blip"] == 1, ["state", "year", "nsdp_manuf_cnst", "g_nsdp_manuf_cnst"]].round(3))

print("\nCorrelation of the two output measures (levels, logs):",
      round(panel[["ln_nsdp_manuf_cnst", "ln_nsdp_manuf_defl"]].corr().iloc[0, 1], 3))
print("Rows flagged suspect_obs = 1:", panel["suspect_obs"].sum(), "of", len(panel))

panel.to_csv("analysis_panel.csv", index=False)
print("\nUpdated analysis_panel.csv with: output_blip, suspect_obs, ln_nsdp_manuf_defl, g_nsdp_manuf_defl")

In [ ]:
# =====================================================================
# STEP 3d: Credit jumps + Andhra Pradesh / Telangana split
# Run AFTER Step 3c (updates analysis_panel.csv)
# =====================================================================
import numpy as np
import pandas as pd
from IPython.display import display

DROP_UNDIVIDED_AP = False     # drop Andhra Pradesh rows before 2013-14 split (FY2013-14 = year 2014)

panel = pd.read_csv("analysis_panel.csv").sort_values(["state", "year"]).reset_index(drop=True)

# ---------- 1) Andhra Pradesh: credit before 2014 includes Telangana ----------
if DROP_UNDIVIDED_AP:
    n0 = len(panel)
    panel = panel[~((panel["state"] == "Andhra Pradesh") & (panel["year"] < 2014))].copy()
    print(f"Dropped {n0 - len(panel)} undivided-Andhra Pradesh rows (before FY2013-14).")
    # first AP year must not use the undivided 2013 value as its lag / growth base
    first_ap = (panel["state"] == "Andhra Pradesh") & (panel["year"] == 2014)
    lag_cols = ["l1_ln_real_cr_out_manuf", "l1_ln_pc_nsdp_cnst"] + \
               [c for c in panel.columns if c.startswith("g_")]
    panel.loc[first_ap, lag_cols] = np.nan

# ---------- 2) Credit jumps and one-year credit blips ----------
panel = panel.sort_values(["state", "year"]).reset_index(drop=True)
g = panel.groupby("state")
consec_next = g["year"].shift(-1) - panel["year"] == 1
consec_prev = panel["year"] - g["year"].shift(1) == 1

gc_now  = panel["g_real_cr_out_manuf"]
gc_next = g["g_real_cr_out_manuf"].shift(-1).where(consec_next)
panel["credit_jump"] = (gc_now.abs() > 0.7).astype(int)
panel["credit_blip"] = ((gc_now.abs() > 0.3) & (gc_next.abs() > 0.3)
                        & (np.sign(gc_now) != np.sign(gc_next))).astype(int)

# the lagged-credit model uses LAST year's credit -> also flag the year after a credit blip
prev_blip = g["credit_blip"].shift(1).where(consec_prev).fillna(0).astype(int)

# ---------- 3) Update the combined robustness flag ----------
panel["suspect_obs"] = ((panel["output_jump"] == 1) | (panel["output_blip"] == 1) |
                        (panel["credit_jump"] == 1) | (panel["credit_blip"] == 1) |
                        (prev_blip == 1)).astype(int)

print("\nCredit jumps (>70%) and one-year credit blips:")
show = panel.loc[(panel["credit_jump"] == 1) | (panel["credit_blip"] == 1),
                 ["state", "year", "cr_out_manuf", "g_real_cr_out_manuf", "credit_jump", "credit_blip"]]
display(show.round(3))

print(f"\nRows flagged suspect_obs = 1: {panel['suspect_obs'].sum()} of {len(panel)}")
print("By state:")
display(panel.groupby("state")["suspect_obs"].sum().loc[lambda s: s > 0].sort_values(ascending=False).to_frame())

panel.to_csv("analysis_panel.csv", index=False)
print("\nUpdated analysis_panel.csv with: credit_jump, credit_blip, suspect_obs (updated)"
      + ("; undivided AP rows removed" if DROP_UNDIVIDED_AP else ""))

In [ ]:
# =====================================================================
# STEP 4: Describe the data before running any regression
# Reads analysis_panel.csv  ->  saves tables/ (csv) and figures/ (png)
# =====================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter, NullFormatter
from pathlib import Path
from scipy import stats
from IPython.display import display

panel = pd.read_csv("analysis_panel.csv").sort_values(["state", "year"])
Path("tables").mkdir(exist_ok=True)
Path("figures").mkdir(exist_ok=True)

# ---------- chart style (colour-blind-safe, one hue per tercile) ----------
TERC_COL  = {1: "#2a78d6", 2: "#1baf7a", 3: "#eb6834"}          # blue / aqua / orange
TERC_NAME = {1: "Credit-scarce (T1)", 2: "Middle (T2)", 3: "Credit-rich (T3)"}
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
# plt.rcParams.update({
#     "figure.dpi": 110, "savefig.dpi": 300, "font.size": 10,
#     "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
#     "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
#     "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
#     "legend.frameon": False,
# })
plt.rcParams.update({
    # --- canvas & export: sized for a LaTeX page (text width ~6.3 in) ---
    "figure.figsize": (6.3, 3.6), "figure.dpi": 120,
    "savefig.dpi": 300, "savefig.bbox": "tight", "savefig.pad_inches": 0.05,
    # --- fonts: serif, to match the paper (STIX ships with matplotlib) ---
    "font.family": "serif", "font.serif": ["STIX Two Text", "STIXGeneral", "DejaVu Serif"],
    "mathtext.fontset": "stix", "font.size": 9,
    "axes.titlesize": 10, "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "legend.fontsize": 8, "axes.titlelocation": "left",
    # --- quiet frame ---
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#9a9a95", "axes.linewidth": 0.6,
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "xtick.major.size": 3, "ytick.major.size": 0, "xtick.major.width": 0.6,
    "axes.labelcolor": "#2b2b2b", "text.color": "#2b2b2b",
    # --- grid: faint, horizontal only, behind the data ---
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#ebebe7",
    "grid.linewidth": 0.6, "axes.axisbelow": True,
    # --- marks ---
    "lines.linewidth": 1.8, "lines.solid_capstyle": "round",
    "legend.frameon": False,
})
def fy(y):                      # 2015 -> "2014-15"
    return f"{int(y) - 1}-{str(int(y))[-2:]}"

# =====================================================================
# 4.1 Summary statistics
# =====================================================================
VARS = {
    "real_cr_out_manuf":        "Real manufacturing credit (Rs crore, 2011-12 prices)",
    "nsdp_manuf_cnst":          "Manufacturing output (Rs crore, 2011-12 prices)",
    "nsdp_manuf_unreg_cnst":    "Unregistered mfg output (Rs crore, to 2014-15)",
    "pc_nsdp_cnst":             "Income per person (Rs, 2011-12 prices)",
    "g_real_cr_out_manuf":      "Growth of real mfg credit (log diff)",
    "g_nsdp_manuf_cnst":        "Growth of mfg output (log diff)",
    "credit_depth_gsdp":        "Credit / GSDP",
    "initial_depth":            "Initial credit depth (credit / GSDP, first year)",
    "manuf_credit_to_output":   "Mfg credit / mfg output",
    "avg_ticket_manuf_lakh":    "Average mfg loan (Rs lakh per account)",
    "small_ticket_share_manuf": "Small-ticket share of mfg credit (<= Rs 25 lakh)",
    "grassroots_share_manuf":   "Grassroots share of mfg credit",
    "util_manuf":               "Utilisation of mfg credit limits",
    "manuf_share":              "Manufacturing share of state output",
}
VARS = {k: v for k, v in VARS.items() if k in panel.columns}
desc = panel[list(VARS)].describe(percentiles=[.25, .5, .75]).T
desc = desc[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]]
desc.insert(0, "Variable", desc.index.map(VARS))
desc["count"] = desc["count"].astype(int)
print(f"4.1 Summary statistics  |  {panel['state'].nunique()} states, "
      f"{fy(panel['year'].min())} to {fy(panel['year'].max())}, {len(panel)} state-years")
display(desc.round(3))
desc.round(4).to_csv("tables/table4_1_summary_stats.csv")

# =====================================================================
# 4.2 All-India trend: real credit vs output (index, first common year = 100)
#     uses only states present in every year so the totals are comparable
# =====================================================================
years = sorted(panel["year"].unique())
full = panel.groupby("state")["year"].nunique()
full_states = full[full == len(years)].index
tot = (panel[panel["state"].isin(full_states)]
       .groupby("year")[["real_cr_out_manuf", "nsdp_manuf_cnst"]].sum())
idx = tot / tot.iloc[0] * 100

fig, ax = plt.subplots(figsize=(7.5, 4))
for col, name, c in [("real_cr_out_manuf", "Real manufacturing credit", TERC_COL[1]),
                     ("nsdp_manuf_cnst", "Manufacturing output", TERC_COL[3])]:
    ax.plot(idx.index, idx[col], color=c, label=name)
    ax.annotate(f"{name}  {idx[col].iloc[-1]:.0f}", (idx.index[-1], idx[col].iloc[-1]),
                xytext=(6, 0), textcoords="offset points", va="center", color=INK, fontsize=9)
ax.axhline(100, color=INK2, lw=0.8, ls="--")
ax.set_xticks(years[::2]); ax.set_xticklabels([fy(y) for y in years[::2]], rotation=45, ha="right")
ax.set_ylabel(f"Index ({fy(years[0])} = 100)")
ax.set_title(f"Real manufacturing credit and output, {len(full_states)} states combined",
             loc="left", fontsize=11, color=INK)
ax.legend(loc="upper left")
ax.set_xlim(years[0], years[-1] + 4.5)
plt.tight_layout(); plt.savefig("figures/fig4_2_india_trend.png", bbox_inches="tight"); plt.show()

# =====================================================================
# 4.3 Selected states: credit vs output (index), two largest states per tercile
# =====================================================================
avg_out = panel.groupby("state").agg(out=("nsdp_manuf_cnst", "mean"), terc=("depth_tercile", "first"))
picks = (avg_out.dropna().sort_values("out", ascending=False)
         .groupby("terc").head(2).sort_values(["terc", "out"], ascending=[True, False]))
# To choose your own states instead, e.g.:
# picks = avg_out.loc[["Bihar", "Assam", "Karnataka", "Uttar Pradesh", "Maharashtra", "Gujarat"]]

fig, axes = plt.subplots(2, 3, figsize=(11, 6), sharey=True)
for ax, (st, row) in zip(axes.T.flat, picks.iterrows()):
    s = panel[panel["state"] == st].set_index("year")[["real_cr_out_manuf", "nsdp_manuf_cnst"]].dropna()
    s = s / s.iloc[0] * 100
    ax.plot(s.index, s["real_cr_out_manuf"], color=TERC_COL[1], label="Real credit")
    ax.plot(s.index, s["nsdp_manuf_cnst"], color=TERC_COL[3], label="Output")
    ax.axhline(100, color=INK2, lw=0.8, ls="--")
    ax.set_title(f"{st}  ·  {TERC_NAME[int(row['terc'])]}", loc="left", fontsize=10, color=INK)
    ax.set_xticks(s.index[::4]); ax.set_xticklabels([fy(y) for y in s.index[::4]], fontsize=8)
for ax in axes[:, 0]:
    ax.set_ylabel("Index (first year = 100)")
axes[0, 0].legend(loc="upper left")
fig.suptitle("Real manufacturing credit vs output in selected states (columns = terciles)",
             x=0.01, ha="left", fontsize=11, color=INK)
plt.tight_layout(); plt.savefig("figures/fig4_3_selected_states.png", bbox_inches="tight"); plt.show()

# =====================================================================
# 4.4 Does faster credit growth go with faster output growth? (by tercile)
#     Left: one dot per state, average annual growth over the whole period
#     Right: every state-year, last year's credit growth vs this year's output growth
# =====================================================================
def long_growth(s, col):
    x = s[["year", col]].dropna()
    return (x[col].iloc[-1] - x[col].iloc[0]) / (x["year"].iloc[-1] - x["year"].iloc[0]) if len(x) > 1 else np.nan

ld = panel.groupby("state").apply(lambda s: pd.Series({
    "g_credit": long_growth(s, "ln_real_cr_out_manuf"),
    "g_output": long_growth(s, "ln_nsdp_manuf_cnst"),
    "terc": s["depth_tercile"].iloc[0]}), include_groups=False).dropna()

gc = "g_real_cr_out_manuf_w" if "g_real_cr_out_manuf_w" in panel else "g_real_cr_out_manuf"
gy = "g_nsdp_manuf_cnst_w" if "g_nsdp_manuf_cnst_w" in panel else "g_nsdp_manuf_cnst"
yr = panel.copy()
consec = yr.groupby("state")["year"].diff() == 1
yr["g_credit_lag"] = yr.groupby("state")[gc].shift(1).where(consec)
yr = yr.dropna(subset=["g_credit_lag", gy, "depth_tercile"])

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5))
slopes = []
for t in [1, 2, 3]:
    d = ld[ld["terc"] == t]
    a1.scatter(d["g_credit"] * 100, d["g_output"] * 100, s=45, color=TERC_COL[t],
               edgecolor="white", linewidth=1, label=TERC_NAME[t], zorder=3)
    for st, r in d.iterrows():
        a1.annotate(st, (r["g_credit"] * 100, r["g_output"] * 100), xytext=(4, 3),
                    textcoords="offset points", fontsize=7, color=INK2)
    e = yr[yr["depth_tercile"] == t]
    a2.scatter(e["g_credit_lag"] * 100, e[gy] * 100, s=14, color=TERC_COL[t], alpha=0.55, linewidth=0)
    if len(e) > 2:
        fit = stats.linregress(e["g_credit_lag"], e[gy])
        xs = np.linspace(e["g_credit_lag"].quantile(.02), e["g_credit_lag"].quantile(.98), 50)
        a2.plot(xs * 100, (fit.intercept + fit.slope * xs) * 100, color=TERC_COL[t],
                label=f"{TERC_NAME[t]}: slope {fit.slope:.2f}")
        slopes.append({"tercile": TERC_NAME[t], "n": len(e), "slope": fit.slope, "p_value": fit.pvalue})
a1.set_xlabel("Average annual growth of real mfg credit (%)")
a1.set_ylabel("Average annual growth of mfg output (%)")
a1.set_title("Whole period, one dot per state", loc="left", fontsize=10, color=INK)
a1.legend(loc="best", fontsize=8)
a2.set_xlabel("Real mfg credit growth, previous year (%)")
a2.set_ylabel("Mfg output growth, this year (%)")
a2.set_title("Year by year, with a simple fitted line per tercile", loc="left", fontsize=10, color=INK)
a2.axhline(0, color=INK2, lw=0.8); a2.axvline(0, color=INK2, lw=0.8)
a2.legend(loc="best", fontsize=8)
plt.tight_layout(); plt.savefig("figures/fig4_4_growth_scatter.png", bbox_inches="tight"); plt.show()
print("4.4 Simple slopes (no controls - a first look only, NOT the regression result):")
display(pd.DataFrame(slopes).round(3))

# =====================================================================
# 4.5 Who borrows in credit-scarce vs credit-rich states? (state averages)
# =====================================================================
PROF = {
    "initial_depth":               "Initial credit depth",
    "pc_nsdp_cnst":                "Income per person (Rs)",
    "manuf_share":                 "Manufacturing share of output",
    "avg_ticket_manuf_lakh":       "Average mfg loan (Rs lakh)",
    "small_ticket_share_manuf":    "Small-ticket share of mfg credit",
    "grassroots_share_manuf":      "Grassroots share of mfg credit",
    "rrb_sfb_share_manuf":         "RRB + SFB share of mfg credit",
    "rural_semiurban_share_manuf": "Rural + semi-urban share of mfg credit",
    "util_manuf":                  "Utilisation of mfg limits",
    "manuf_credit_to_output":      "Mfg credit / mfg output",
}
PROF = {k: v for k, v in PROF.items() if k in panel.columns}
st_mean = panel.groupby("state")[list(PROF) + ["depth_tercile"]].mean().dropna(subset=["depth_tercile"])
prof = st_mean.groupby("depth_tercile")[list(PROF)].mean().T
prof.columns = [TERC_NAME[int(c)] for c in prof.columns]
t1, t3 = st_mean[st_mean["depth_tercile"] == 1], st_mean[st_mean["depth_tercile"] == 3]
prof["T1 - T3"] = prof.iloc[:, 0] - prof.iloc[:, -1]
prof["p-value (T1 vs T3)"] = [stats.ttest_ind(t1[v].dropna(), t3[v].dropna(), equal_var=False).pvalue
                              if t1[v].notna().sum() > 1 and t3[v].notna().sum() > 1 else np.nan for v in PROF]
prof.index = [PROF[v] for v in prof.index]
print("4.5 Profile of states by initial credit depth (average of state averages; Welch t-test T1 vs T3)")
display(prof.round(3))
prof.round(4).to_csv("tables/table4_5_profile_by_tercile.csv")

share_vars = [v for v in ["small_ticket_share_manuf", "grassroots_share_manuf",
                          "rrb_sfb_share_manuf", "rural_semiurban_share_manuf"] if v in PROF]
m = st_mean.groupby("depth_tercile")[share_vars].mean()
fig, ax = plt.subplots(figsize=(8.5, 4))
w = 0.26
x = np.arange(len(share_vars))
for i, t in enumerate(m.index):
    bars = ax.bar(x + (i - 1) * w, m.loc[t] * 100, width=w - 0.02, color=TERC_COL[int(t)],
                  label=TERC_NAME[int(t)], zorder=3)
    ax.bar_label(bars, fmt="%.0f", fontsize=8, color=INK2, padding=2)
ax.set_xticks(x); ax.set_xticklabels([PROF[v].replace(" of mfg credit", "") for v in share_vars])
ax.set_ylabel("% of manufacturing credit")
ax.set_title("Small-borrower profile of manufacturing credit, by initial credit depth",
             loc="left", fontsize=11, color=INK)
ax.set_ylim(0, m.values.max() * 100 * 1.25)
ax.legend(loc="upper left", ncol=3, fontsize=9); ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.savefig("figures/fig4_5_borrower_profile.png", bbox_inches="tight"); plt.show()

# =====================================================================
# 4.6 Do poorer states start with less credit? (the Sita vs Ramesh picture)
# =====================================================================
first = panel.sort_values("year").groupby("state").first()
first = first.dropna(subset=["initial_depth", "ln_pc_nsdp_cnst", "depth_tercile"])
fig, ax = plt.subplots(figsize=(7.5, 5))
for t in [1, 2, 3]:
    d = first[first["depth_tercile"] == t]
    ax.scatter(np.exp(d["ln_pc_nsdp_cnst"]) / 1000, d["initial_depth"], s=45, color=TERC_COL[t],
               edgecolor="white", linewidth=1, label=TERC_NAME[t], zorder=3)
    for st, r in d.iterrows():
        ax.annotate(st, (np.exp(r["ln_pc_nsdp_cnst"]) / 1000, r["initial_depth"]), xytext=(4, 3),
                    textcoords="offset points", fontsize=7, color=INK2)
r_ = stats.pearsonr(first["ln_pc_nsdp_cnst"], first["initial_depth"])
ax.set_xscale("log")
ax.xaxis.set_major_formatter(ScalarFormatter()); ax.xaxis.set_minor_formatter(NullFormatter())
ax.set_xticks([t for t in [20, 30, 40, 60, 80, 100, 150, 200, 300, 400]
               if np.exp(first["ln_pc_nsdp_cnst"]).min() / 1000 * 0.9 <= t <= np.exp(first["ln_pc_nsdp_cnst"]).max() / 1000 * 1.1])
ax.set_xlabel("Income per person in first year (Rs '000, 2011-12 prices, log scale)")
ax.set_ylabel("Initial credit depth (credit / GSDP)")
ax.set_title(f"Initial income vs initial credit depth (correlation {r_[0]:.2f}, p = {r_[1]:.3f})",
             loc="left", fontsize=11, color=INK)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=3, fontsize=8)
plt.tight_layout(); plt.savefig("figures/fig4_6_income_vs_depth.png", bbox_inches="tight"); plt.show()

# =====================================================================
# 4.7 Correlations between the main regression variables
# =====================================================================
CORR = {
    "ln_nsdp_manuf_cnst":       "ln output",
    "l1_ln_real_cr_out_manuf":  "ln credit (t-1)",
    "l1_ln_pc_nsdp_cnst":       "ln income pc (t-1)",
    "initial_depth":            "initial depth",
    "small_ticket_share_manuf": "small-ticket share",
    "grassroots_share_manuf":   "grassroots share",
    "g_nsdp_manuf_cnst":        "output growth",
    "g_real_cr_out_manuf":      "credit growth",
}
CORR = {k: v for k, v in CORR.items() if k in panel.columns}
corr = panel[list(CORR)].corr().rename(index=CORR, columns=CORR)
print("4.7 Correlation matrix (pooled state-years)")
display(corr.round(2))
corr.round(3).to_csv("tables/table4_7_correlations.csv")

print("\nSaved: tables/table4_1_summary_stats.csv, table4_5_profile_by_tercile.csv, table4_7_correlations.csv")
print("       figures/fig4_2 ... fig4_6 (.png, 300 dpi)")

In [ ]:
# =====================================================================
# STEP 5: Main test of H1 - does last year's credit raise this year's output?
# Two-way fixed effects (state + year), standard errors clustered by state
# Reads analysis_panel.csv  ->  saves tables/table5_h1_main.csv
# =====================================================================
import sys, subprocess
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from IPython.display import display

try:
    from linearmodels.panel import PanelOLS
except ImportError:                       # installs once, then continues
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "linearmodels"])
    from linearmodels.panel import PanelOLS

Path("tables").mkdir(exist_ok=True)
df = pd.read_csv("analysis_panel.csv")

Y     = "ln_nsdp_manuf_cnst"          # log real manufacturing output
X     = "l1_ln_real_cr_out_manuf"     # log real manufacturing credit, last year
CTRL  = ["l1_ln_pc_nsdp_cnst"]        # log income per person, last year
G_Y, G_X = "g_nsdp_manuf_cnst", "g_real_cr_out_manuf"   # growth versions

# ---------- estimation sample: same rows for every column of the table ----------
est = df.dropna(subset=[Y, X] + CTRL).copy()
est = est.set_index(["state", "year"]).sort_index()
print(f"Estimation sample: {len(est)} state-years, {est.index.get_level_values(0).nunique()} states, "
      f"{est.index.get_level_values(1).min()}-{est.index.get_level_values(1).max()}")

def fit(y, xs, data, entity=True, time=True, cov="clustered"):
    d = data.dropna(subset=[y] + xs)
    exog = d[xs].assign(const=1.0)
    mod = PanelOLS(d[y], exog, entity_effects=entity, time_effects=time, drop_absorbed=True)
    if cov == "clustered":
        return mod.fit(cov_type="clustered", cluster_entity=True)
    return mod.fit(cov_type="kernel")      # Driscoll-Kraay: robust to shocks shared across states

# ---------- growth version: last year's credit growth -> this year's output growth ----------
gro = df.sort_values(["state", "year"]).copy()
consec = gro.groupby("state")["year"].diff() == 1
gro["l1_g_credit"] = gro.groupby("state")[G_X].shift(1).where(consec)
gro = gro.set_index(["state", "year"])

models = {
    "(1) Pooled OLS":            fit(Y, [X],        est, entity=False, time=False),
    "(2) State FE":              fit(Y, [X],        est, entity=True,  time=False),
    "(3) State + Year FE":       fit(Y, [X],        est, entity=True,  time=True),
    "(4) + Income control":      fit(Y, [X] + CTRL, est, entity=True,  time=True),   # MAIN
    "(5) Growth on growth":      fit(G_Y, ["l1_g_credit"], gro, entity=True, time=True),
}
MAIN = "(4) + Income control"

# ---------- results table: coefficient, stars, (standard error) ----------
LABEL = {X: "ln real mfg credit (t-1)", "l1_g_credit": "Real mfg credit growth (t-1)",
         "l1_ln_pc_nsdp_cnst": "ln income per person (t-1)"}
def stars(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""

rows = {}
for name, r in models.items():
    col = {}
    for v in [X, "l1_g_credit", "l1_ln_pc_nsdp_cnst"]:
        if v in r.params.index:
            col[LABEL[v]] = f"{r.params[v]:.3f}{stars(r.pvalues[v])}"
            col[LABEL[v] + " (s.e.)"] = f"({r.std_errors[v]:.3f})"
    col["State fixed effects"] = "No" if name.startswith("(1)") else "Yes"
    col["Year fixed effects"]  = "No" if name.startswith(("(1)", "(2)")) else "Yes"
    col["Observations"] = f"{int(r.nobs)}"
    col["States"] = f"{int(r.entity_info['total'])}"
    col["R-squared (within)"] = f"{r.rsquared:.3f}"    # fit after removing the fixed effects
    rows[name] = col
order = [LABEL[X], LABEL[X] + " (s.e.)", LABEL["l1_g_credit"], LABEL["l1_g_credit"] + " (s.e.)",
         LABEL["l1_ln_pc_nsdp_cnst"], LABEL["l1_ln_pc_nsdp_cnst"] + " (s.e.)",
         "State fixed effects", "Year fixed effects", "Observations", "States", "R-squared (within)"]
table = pd.DataFrame(rows).reindex(order).fillna("")
table.index = [i if "(s.e.)" not in i else "" for i in table.index]
print("\nTable 5. Effect of manufacturing credit on manufacturing output")
print("Dependent variable: ln real manufacturing output (columns 1-4); output growth (column 5)")
print("Standard errors clustered by state in brackets. *** p<0.01, ** p<0.05, * p<0.10")
display(table)
table.to_csv("tables/table5_h1_main.csv")

# ---------- extra checks on the MAIN column ----------
main = models[MAIN]
b, se, p2 = main.params[X], main.std_errors[X], main.pvalues[X]
p1 = p2 / 2 if b > 0 else 1 - p2 / 2                     # one-sided test of H1: beta > 0
dk = fit(Y, [X] + CTRL, est, entity=True, time=True, cov="kernel")

# Wild cluster bootstrap (few clusters ~30 -> more reliable p-value). Null imposed, Rademacher weights.
def wild_cluster_p(data, y, x, ctrls, reps=1999, seed=42):
    d = data.dropna(subset=[y, x] + ctrls).reset_index()
    Z = pd.get_dummies(d[["state", "year"]].astype(str), drop_first=True, dtype=float)
    Z = np.column_stack([np.ones(len(d)), Z.values, d[ctrls].values])
    M = np.eye(len(d)) - Z @ np.linalg.pinv(Z)            # partials out FE + controls
    xt, yt = M @ d[x].values, M @ d[y].values
    g = pd.factorize(d["state"])[0]; G = g.max() + 1
    def tstat(yy):
        beta = xt @ yy / (xt @ xt)
        e = yy - beta * xt
        score = np.bincount(g, weights=xt * e, minlength=G)
        n, k = len(d), Z.shape[1] + 1
        V = (G / (G - 1)) * ((n - 1) / (n - k)) * (score @ score) / (xt @ xt) ** 2
        return beta / np.sqrt(V)
    t0 = tstat(yt)
    rng = np.random.default_rng(seed)
    tb = np.array([tstat(M @ (yt * rng.choice([-1.0, 1.0], G)[g])) for _ in range(reps)])
    return t0, np.mean(np.abs(tb) >= abs(t0))

t_wcb, p_wcb = wild_cluster_p(est, Y, X, CTRL)

print("\nChecks on the MAIN model (column 4):")
checks = pd.DataFrame({
    "beta (credit elasticity)": [b, b, b],
    "standard error": [se, dk.std_errors[X], np.nan],
    "p-value (two-sided)": [p2, dk.pvalues[X], p_wcb],
}, index=["Clustered by state (main)", "Driscoll-Kraay (shared shocks)", "Wild cluster bootstrap"])
display(checks.round(4))

# ---------- plain-English reading ----------
mean_out = np.exp(est[Y]).mean()
print("\nWHAT THIS MEANS")
print(f"  - A 10% rise in last year's real manufacturing credit goes with a {b*10:.2f}% change in this "
      f"year's manufacturing output, comparing each state with itself over time.")
print(f"  - For a state with average output (about Rs {mean_out:,.0f} crore), that is roughly "
      f"Rs {mean_out * b * 0.10:,.0f} crore of output per year.")
verdict = ("SUPPORTED: beta > 0 and significant at 5% (one-sided)" if (b > 0 and p1 < 0.05) else
           "WEAKLY supported: beta > 0, significant only at 10% (one-sided)" if (b > 0 and p1 < 0.10) else
           "NOT supported: beta is not significantly above zero")
print(f"  - H1 (beta > 0): one-sided p = {p1:.4f}  ->  {verdict}")
print(f"  - Pooled OLS (col 1) vs fixed effects (col 4): {models['(1) Pooled OLS'].params[X]:.3f} vs {b:.3f}. "
      "The gap shows how much of the raw link was just 'big states have more of both'.")

In [ ]:
# =====================================================================
# STEP 5b: Why is the H1 effect zero? Pre-planned diagnostic checks
# Same model as Table 5 column (4) unless stated. Reads analysis_panel.csv
# =====================================================================
import sys, subprocess
import numpy as np
import pandas as pd
import statsmodels.api as sm
from pathlib import Path
from scipy import stats
from IPython.display import display
try:
    from linearmodels.panel import PanelOLS
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "linearmodels"])
    from linearmodels.panel import PanelOLS

Path("tables").mkdir(exist_ok=True)
df = pd.read_csv("analysis_panel.csv").sort_values(["state", "year"]).reset_index(drop=True)
g = df.groupby("state")

def lag(col, k):
    """value k years earlier, only if that exact year exists"""
    s = g[col].shift(k)
    return s.where(df["year"] - g["year"].shift(k) == k)

CR = "ln_real_cr_out_manuf"
df["l1_cr"] = lag(CR, 1); df["l2_cr"] = lag(CR, 2); df["l3_cr"] = lag(CR, 3)
df["avg3_cr"] = df[["l1_cr", "l2_cr", "l3_cr"]].mean(axis=1, skipna=False)   # avg of t-1..t-3
df["l1_tot"] = lag("ln_real_cr_out_total", 1) if "ln_real_cr_out_total" in df else np.nan
df["w_size"] = g["nsdp_manuf_cnst"].transform("mean")                          # state size weight

Y, CTRL = "ln_nsdp_manuf_cnst", ["l1_ln_pc_nsdp_cnst"]

def fe(y, xs, data, weights=None, combo=None):
    d = data.dropna(subset=[y] + xs + CTRL + ([weights] if weights else [])).set_index(["state", "year"])
    mod = PanelOLS(d[y], d[xs + CTRL].assign(const=1.0), entity_effects=True, time_effects=True,
                   weights=d[weights] if weights else None, drop_absorbed=True)
    r = mod.fit(cov_type="clustered", cluster_entity=True)
    if combo:                                   # sum of several lag coefficients (cumulative effect)
        w = np.array([1.0 if v in combo else 0.0 for v in r.params.index])
        b = float(w @ r.params.values); se = float(np.sqrt(w @ r.cov.values @ w))
        p = 2 * (1 - stats.t.cdf(abs(b / se), df=r.entity_info["total"] - 1))
    else:
        b, se, p = r.params[xs[0]], r.std_errors[xs[0]], r.pvalues[xs[0]]
    return dict(beta=b, se=se, p=p, N=int(r.nobs), states=int(r.entity_info["total"]))

specs = []
def add(name, why, res):
    specs.append({"Check": name, "Why we try it": why, **res})

add("A. Main (Table 5 col 4)", "reference", fe(Y, ["l1_cr"], df))
add("B. Credit 2 years earlier", "machines take time to install", fe(Y, ["l2_cr"], df))
add("C. Credit 3 years earlier", "longer build-up", fe(Y, ["l3_cr"], df))
add("D. Average credit, t-1 to t-3", "smooths one-year blips in credit", fe(Y, ["avg3_cr"], df))
add("E. Cumulative: lags 1+2+3", "total effect over 3 years", fe(Y, ["l1_cr", "l2_cr", "l3_cr"], df,
    combo=["l1_cr", "l2_cr", "l3_cr"]))
add("F. Drop suspect rows", "blips/jumps add noise", fe(Y, ["l1_cr"], df[df["suspect_obs"] == 0]))
add("G. Drop small mfg states", "tiny NE states are very noisy", fe(Y, ["l1_cr"], df[df["small_manuf_state"] == 0]))
add("H. Drop Maharashtra & Delhi", "credit booked at HQ branches", fe(Y, ["l1_cr"], df[df["branch_hub"] == 0]))
add("I. Weight by state size", "big states measured more accurately", fe(Y, ["l1_cr"], df, weights="w_size"))
add("J. Output = current price / state deflator", "avoids constant-price breaks",
    fe("ln_nsdp_manuf_defl", ["l1_cr"], df))
if df["l1_tot"].notna().any():
    add("K. Total credit instead of mfg credit", "mfg credit may be mis-classified", fe(Y, ["l1_tot"], df))

# L. Long difference: change between first 3 and last 3 years, one row per state
def window(s, first=True):
    s = s.dropna(subset=[Y, CR, "ln_pc_nsdp_cnst"])
    w = s.head(3) if first else s.tail(3)
    return w[[Y, CR, "ln_pc_nsdp_cnst"]].mean()
ld = pd.DataFrame({st: window(s, False) - window(s, True) for st, s in df.groupby("state")
                   if s[[Y, CR]].dropna().shape[0] >= 8}).T
ols = sm.OLS(ld[Y], sm.add_constant(ld[[CR, "ln_pc_nsdp_cnst"]])).fit(cov_type="HC1")
add("L. Long difference (start vs end)", "slow, long-run effect; ignores yearly noise",
    dict(beta=ols.params[CR], se=ols.bse[CR], p=ols.pvalues[CR], N=int(ols.nobs), states=int(ols.nobs)))

res = pd.DataFrame(specs)
res["sig"] = res["p"].map(lambda p: "***" if p < .01 else "**" if p < .05 else "*" if p < .10 else "")
print("Table 5b. Diagnostic checks on the H1 credit effect (beta = credit elasticity of output)")
print("All FE models: state + year FE, income control, SE clustered by state.")
display(res.round(3))
res.round(4).to_csv("tables/table5b_h1_diagnostics.csv", index=False)

n_pos = ((res["beta"] > 0) & (res["p"] < 0.10)).sum()
print(f"\nPositive and significant (10%) in {n_pos} of {len(res)} checks.")

In [ ]:
# =====================================================================
# STEP 6: Test H2 - is credit's effect bigger in credit-scarce states?
# Same base model as Table 5 col (4): state + year FE, income control,
# SE clustered by state. Reads analysis_panel.csv -> tables/ + figures/
# =====================================================================
import sys, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from IPython.display import display
try:
    from linearmodels.panel import PanelOLS
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "linearmodels"])
    from linearmodels.panel import PanelOLS

Path("tables").mkdir(exist_ok=True); Path("figures").mkdir(exist_ok=True)
df = pd.read_csv("analysis_panel.csv").sort_values(["state", "year"]).reset_index(drop=True)

Y, X, CTRL = "ln_nsdp_manuf_cnst", "l1_ln_real_cr_out_manuf", ["l1_ln_pc_nsdp_cnst"]
df = df.dropna(subset=[Y, X, "initial_depth", "depth_tercile"] + CTRL).copy()
df["depth_tercile"] = df["depth_tercile"].astype(int)

# ---------- interaction variables ----------
mean_depth = df.groupby("state")["initial_depth"].first().mean()
df["depth_c"] = df["initial_depth"] - mean_depth               # centred: main effect = effect at AVERAGE depth
df["cr_x_depth"] = df[X] * df["depth_c"]
df["cr_x_low"] = df[X] * (df["depth_tercile"] == 1)
for t in [1, 2, 3]:
    df[f"cr_T{t}"] = df[X] * (df["depth_tercile"] == t)
df["w_size"] = df.groupby("state")["nsdp_manuf_cnst"].transform("mean")

def fe(y, xs, data, weights=None):
    d = data.dropna(subset=[y] + xs + CTRL).set_index(["state", "year"])
    mod = PanelOLS(d[y], d[xs + CTRL].assign(const=1.0), entity_effects=True, time_effects=True,
                   weights=d[weights] if weights else None, drop_absorbed=True)
    return mod.fit(cov_type="clustered", cluster_entity=True)

def lincom(r, w):
    """estimate, s.e., p for a weighted sum of coefficients, e.g. b1 - b3"""
    v = np.array([w.get(k, 0.0) for k in r.params.index])
    b = float(v @ r.params.values); se = float(np.sqrt(v @ r.cov.values @ v))
    p = 2 * (1 - stats.t.cdf(abs(b / se), df=r.entity_info["total"] - 1))
    return b, se, p

# Wild cluster bootstrap p-value for ONE coefficient (null imposed) - reliable with ~30 states
def wild_p(data, y, x, others, reps=1999, seed=42):
    d = data.dropna(subset=[y, x] + others).reset_index(drop=True)
    Z = pd.get_dummies(d[["state", "year"]].astype(str), drop_first=True, dtype=float)
    Z = np.column_stack([np.ones(len(d)), Z.values, d[others].values])
    M = np.eye(len(d)) - Z @ np.linalg.pinv(Z)
    xt, yt = M @ d[x].values, M @ d[y].values
    gid = pd.factorize(d["state"])[0]; G = gid.max() + 1; n, k = len(d), Z.shape[1] + 1
    def t(yy):
        b = xt @ yy / (xt @ xt); e = yy - b * xt
        s = np.bincount(gid, weights=xt * e, minlength=G)
        return b / np.sqrt((G / (G - 1)) * ((n - 1) / (n - k)) * (s @ s) / (xt @ xt) ** 2)
    t0 = t(yt); rng = np.random.default_rng(seed)
    tb = np.array([t(M @ (yt * rng.choice([-1.0, 1.0], G)[gid])) for _ in range(reps)])
    return np.mean(np.abs(tb) >= abs(t0))

def stars(p):
    return "***" if p < .01 else "**" if p < .05 else "*" if p < .10 else ""
def cell(b, se, p):
    return f"{b:.3f}{stars(p)}", f"({se:.3f})"

# =====================================================================
# 6.1 Main H2 table
# =====================================================================
m1 = fe(Y, [X, "cr_x_depth"], df)                     # continuous interaction
m2 = fe(Y, [X, "cr_x_low"], df)                       # T1 dummy interaction
m3 = fe(Y, ["cr_T1", "cr_T2", "cr_T3"], df)           # one slope per tercile
m4 = fe(Y, [X, "cr_x_depth"], df, weights="w_size")   # (1) weighted by state size
m5 = fe(Y, [X, "cr_x_depth"], df[df["small_manuf_state"] == 0])   # (1) without tiny states

T = {}
def put(col, label, b, se, p):
    T.setdefault(col, {})[label], T[col][label + " "] = cell(b, se, p)

for col, r in [("(1) x initial depth", m1), ("(4) weighted by size", m4), ("(5) drop small states", m5)]:
    put(col, "Credit (t-1) at average depth", r.params[X], r.std_errors[X], r.pvalues[X])
    put(col, "Credit x initial depth", r.params["cr_x_depth"], r.std_errors["cr_x_depth"], r.pvalues["cr_x_depth"])
put("(2) x credit-scarce", "Credit (t-1): T2 + T3 states", m2.params[X], m2.std_errors[X], m2.pvalues[X])
put("(2) x credit-scarce", "Extra effect in T1 states", m2.params["cr_x_low"], m2.std_errors["cr_x_low"], m2.pvalues["cr_x_low"])
put("(2) x credit-scarce", "Total effect in T1 states", *lincom(m2, {X: 1, "cr_x_low": 1}))
for t in [1, 2, 3]:
    put("(3) slope per tercile", f"Credit effect in T{t}", m3.params[f"cr_T{t}"], m3.std_errors[f"cr_T{t}"], m3.pvalues[f"cr_T{t}"])
put("(3) slope per tercile", "T1 minus T3", *lincom(m3, {"cr_T1": 1, "cr_T3": -1}))
for col, r in [("(1) x initial depth", m1), ("(2) x credit-scarce", m2), ("(3) slope per tercile", m3),
               ("(4) weighted by size", m4), ("(5) drop small states", m5)]:
    T[col]["Observations"] = str(int(r.nobs)); T[col]["States"] = str(int(r.entity_info["total"]))

order = ["Credit (t-1) at average depth", "Credit x initial depth",
         "Credit (t-1): T2 + T3 states", "Extra effect in T1 states", "Total effect in T1 states",
         "Credit effect in T1", "Credit effect in T2", "Credit effect in T3", "T1 minus T3",
         "Observations", "States"]
order = [o for k in order for o in ([k, k + " "] if k not in ("Observations", "States") else [k])]
table = pd.DataFrame(T)[["(1) x initial depth", "(2) x credit-scarce", "(3) slope per tercile",
                         "(4) weighted by size", "(5) drop small states"]].reindex(order).fillna("")
table.index = [i if not i.endswith(" ") else "" for i in table.index]
print("Table 6. Does the credit effect depend on initial credit depth? (H2)")
print("Dep. var: ln real mfg output. All: state + year FE, ln income pc (t-1). SE clustered by state.")
print("Initial depth is centred at its mean. *** p<0.01, ** p<0.05, * p<0.10")
display(table)
table.to_csv("tables/table6_h2.csv")

# =====================================================================
# 6.2 Split samples: run the H1 model separately for T1 and T3
# =====================================================================
split = []
for t in [1, 2, 3]:
    r = fe(Y, [X], df[df["depth_tercile"] == t])
    split.append({"Group": f"T{t}", "States": int(r.entity_info["total"]), "N": int(r.nobs),
                  "beta": r.params[X], "se": r.std_errors[X], "p": r.pvalues[X]})
split = pd.DataFrame(split)
z = (split.loc[0, "beta"] - split.loc[2, "beta"]) / np.sqrt(split.loc[0, "se"]**2 + split.loc[2, "se"]**2)
print("\n6.2 Separate regressions by tercile (year effects estimated within each group)")
display(split.round(3))
print(f"  T1 vs T3 difference: {split.loc[0,'beta'] - split.loc[2,'beta']:.3f}, "
      f"z = {z:.2f}, p = {2 * (1 - stats.norm.cdf(abs(z))):.3f}")
split.round(4).to_csv("tables/table6b_h2_split.csv", index=False)

# =====================================================================
# 6.3 Picture: credit effect at different levels of initial depth (from model 1)
# =====================================================================
grid = np.linspace(df.groupby("state")["initial_depth"].first().min(),
                   df.groupby("state")["initial_depth"].first().max(), 60)
eff, lo, hi = [], [], []
crit = stats.t.ppf(0.975, m1.entity_info["total"] - 1)
for dpt in grid:
    b, se, _ = lincom(m1, {X: 1, "cr_x_depth": dpt - mean_depth})
    eff.append(b); lo.append(b - crit * se); hi.append(b + crit * se)

INK, INK2, BLUE = "#0b0b0b", "#52514e", "#2a78d6"
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.fill_between(grid, lo, hi, color=BLUE, alpha=0.15, linewidth=0, label="95% confidence band")
ax.plot(grid, eff, color=BLUE, lw=2, label="Estimated credit effect")
ax.axhline(0, color=INK2, lw=0.8)
st_depth = df.groupby("state")["initial_depth"].first()
ax.plot(st_depth, np.full(len(st_depth), ax.get_ylim()[0]), "|", color=INK2, ms=10, alpha=0.7)  # where states sit
for name in ["Bihar", "Maharashtra"]:
    if name in st_depth:
        ax.annotate(name, (st_depth[name], ax.get_ylim()[0]), xytext=(0, 12), textcoords="offset points",
                    ha="center", fontsize=8, color=INK2)
ax.set_xlabel("Initial credit depth (credit / GSDP in first year)")
ax.set_ylabel("Effect of 1% more credit on output (%)")
ax.set_title("Credit effect by initial credit depth (H2 predicts a downward slope)",
             loc="left", fontsize=11, color=INK)
ax.spines[["top", "right"]].set_visible(False); ax.grid(color="#e4e3df", lw=0.6)
ax.legend(frameon=False, fontsize=8, loc="upper right")
plt.tight_layout(); plt.savefig("figures/fig6_marginal_effect.png", dpi=300, bbox_inches="tight"); plt.show()

# =====================================================================
# 6.4 Plain-English verdict
# =====================================================================
b3, p3 = m1.params["cr_x_depth"], m1.pvalues["cr_x_depth"]
p3_one = p3 / 2 if b3 < 0 else 1 - p3 / 2                  # H2 predicts a NEGATIVE interaction
p3_wild = wild_p(df, Y, "cr_x_depth", [X] + CTRL)
d13, _, p13 = lincom(m3, {"cr_T1": 1, "cr_T3": -1})
sd_depth = st_depth.std()

print("\nWHAT THIS MEANS")
print(f"  - At average initial depth ({mean_depth:.2f}), the credit effect is {m1.params[X]:.3f} "
      f"(p = {m1.pvalues[X]:.3f}).")
print(f"  - Interaction = {b3:.3f}: a state with initial depth one standard deviation ({sd_depth:.2f}) LOWER "
      f"has a credit effect {(-b3 * sd_depth):+.3f} different.")
print(f"  - Credit effect: T1 (credit-scarce) {m3.params['cr_T1']:.3f}, T2 {m3.params['cr_T2']:.3f}, "
      f"T3 (credit-rich) {m3.params['cr_T3']:.3f}; T1 - T3 = {d13:.3f} (p = {p13:.3f}).")
print(f"  - H2 test (interaction < 0): one-sided p = {p3_one:.4f}; wild-bootstrap two-sided p = {p3_wild:.4f}")
verdict = ("SUPPORTED: credit helps significantly more where credit was scarce" if (b3 < 0 and p3_one < 0.05) else
           "WEAKLY supported (10% level only)" if (b3 < 0 and p3_one < 0.10) else
           "NOT supported: no significant difference between credit-scarce and credit-rich states")
print(f"  - Verdict: {verdict}")

In [ ]:
# =====================================================================
# STEP 6b: Is the interaction driven by a few extreme states?
# Run in the SAME notebook right after Step 6 (uses df, fe, wild_p, X, Y, CTRL)
# =====================================================================
st = (df.groupby("state")[["initial_depth", "depth_tercile", "small_manuf_state"]].first()
        .sort_values("initial_depth", ascending=False))
print("States by initial credit depth (highest first):")
display(st.round(3))

rows = []
# (a) rank instead of raw depth: every state counts equally, extremes lose their pull
rank = df.groupby("state")["initial_depth"].first().rank(pct=True)
df["rank_c"] = df["state"].map(rank) - rank.mean()
df["cr_x_rank"] = df[X] * df["rank_c"]
r = fe(Y, [X, "cr_x_rank"], df)
rows.append(["Depth as percentile rank", r.params["cr_x_rank"], r.std_errors["cr_x_rank"],
             r.pvalues["cr_x_rank"], wild_p(df, Y, "cr_x_rank", [X] + CTRL), int(r.entity_info["total"])])
# (b) log depth: compresses very high values
ld = np.log(df.groupby("state")["initial_depth"].first())
df["ldep_c"] = df["state"].map(ld) - ld.mean()
df["cr_x_ldep"] = df[X] * df["ldep_c"]
r = fe(Y, [X, "cr_x_ldep"], df)
rows.append(["Depth in logs", r.params["cr_x_ldep"], r.std_errors["cr_x_ldep"],
             r.pvalues["cr_x_ldep"], wild_p(df, Y, "cr_x_ldep", [X] + CTRL), int(r.entity_info["total"])])
# (c) drop the 3 states with the highest initial depth
top3 = st.index[:3].tolist()
d3 = df[~df["state"].isin(top3)]
r = fe(Y, [X, "cr_x_depth"], d3)
rows.append([f"Drop top-3 depth ({', '.join(top3)})", r.params["cr_x_depth"], r.std_errors["cr_x_depth"],
             r.pvalues["cr_x_depth"], wild_p(d3, Y, "cr_x_depth", [X] + CTRL), int(r.entity_info["total"])])

out = pd.DataFrame(rows, columns=["Version of Table 6 col (1)", "interaction", "se",
                                  "p (clustered)", "p (wild bootstrap)", "states"])
print("\nInteraction (credit x initial depth) under alternative depth measures. H2 predicts < 0.")
display(out.round(3))
out.round(4).to_csv("tables/table6c_h2_leverage.csv", index=False)

In [ ]:
# =====================================================================
# STEP 7: The small-firm channel - does credit that reaches SMALL borrowers matter?
# Same setup as Steps 5-6: state + year FE, ln income pc (t-1), SE clustered by state
# Reads analysis_panel.csv -> tables/table7a_channel.csv, table7b_informal.csv
# =====================================================================
import sys, subprocess
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from IPython.display import display
try:
    from linearmodels.panel import PanelOLS
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "linearmodels"])
    from linearmodels.panel import PanelOLS

Path("tables").mkdir(exist_ok=True)
df = pd.read_csv("analysis_panel.csv").sort_values(["state", "year"]).reset_index(drop=True)
g = df.groupby("state")
consec = df["year"] - g["year"].shift(1) == 1
def lag1(s):
    return s.groupby(df["state"]).shift(1).where(consec)
def ln(s):
    return np.log(s.where(s > 0))

# ---------- build the credit pieces (Rs crore, 2011-12 prices) ----------
real_cr = np.exp(df["ln_real_cr_out_manuf"])
df["small_cr"] = real_cr * df["small_ticket_share_manuf"]          # loans in cells averaging <= Rs 25 lakh
df["large_cr"] = real_cr * (1 - df["small_ticket_share_manuf"])    # everything else
df["grass_cr"] = real_cr * df["grassroots_share_manuf"]            # RRB/SFB + rural & semi-urban branches

df["l1_ln_small"] = lag1(ln(df["small_cr"]))
df["l1_ln_large"] = lag1(ln(df["large_cr"]))
df["l1_ln_grass"] = lag1(ln(df["grass_cr"]))
df["l1_small_share"] = lag1(df["small_ticket_share_manuf"])
df["l1_grass_share"] = lag1(df["grassroots_share_manuf"])
X, CTRL = "l1_ln_real_cr_out_manuf", ["l1_ln_pc_nsdp_cnst"]
m_share = df["l1_small_share"].mean()
df["cr_x_small_share"] = df[X] * (df["l1_small_share"] - m_share)  # centred interaction

def fe(y, xs, data):
    d = data.dropna(subset=[y] + xs + CTRL).set_index(["state", "year"])
    return PanelOLS(d[y], d[xs + CTRL].assign(const=1.0), entity_effects=True, time_effects=True,
                    drop_absorbed=True).fit(cov_type="clustered", cluster_entity=True)

def lincom(r, w):
    v = np.array([w.get(k, 0.0) for k in r.params.index])
    b = float(v @ r.params.values); se = float(np.sqrt(v @ r.cov.values @ v))
    return b, se, 2 * (1 - stats.t.cdf(abs(b / se), df=r.entity_info["total"] - 1))

def wild_p(data, y, x, others, reps=1999, seed=42):
    d = data.dropna(subset=[y, x] + others).reset_index(drop=True)
    Z = pd.get_dummies(d[["state", "year"]].astype(str), drop_first=True, dtype=float)
    Z = np.column_stack([np.ones(len(d)), Z.values, d[others].values])
    M = np.eye(len(d)) - Z @ np.linalg.pinv(Z)
    xt, yt = M @ d[x].values, M @ d[y].values
    gid = pd.factorize(d["state"])[0]; G = gid.max() + 1; n, k = len(d), Z.shape[1] + 1
    def t(yy):
        b = xt @ yy / (xt @ xt); e = yy - b * xt
        s = np.bincount(gid, weights=xt * e, minlength=G)
        return b / np.sqrt((G / (G - 1)) * ((n - 1) / (n - k)) * (s @ s) / (xt @ xt) ** 2)
    t0 = t(yt); rng = np.random.default_rng(seed)
    tb = np.array([t(M @ (yt * rng.choice([-1.0, 1.0], G)[gid])) for _ in range(reps)])
    return np.mean(np.abs(tb) >= abs(t0))

def stars(p):
    return "***" if p < .01 else "**" if p < .05 else "*" if p < .10 else ""

LABEL = {X: "ln total mfg credit (t-1)", "l1_ln_small": "ln small-ticket credit (t-1)",
         "l1_ln_large": "ln large-ticket credit (t-1)", "l1_ln_grass": "ln grassroots credit (t-1)",
         "l1_small_share": "Small-ticket share (t-1)", "l1_grass_share": "Grassroots share (t-1)",
         "cr_x_small_share": "Total credit x small-ticket share", "l1_ln_pc_nsdp_cnst": "ln income pc (t-1)"}

def make_table(models, extra=None):
    T = {}
    for name, r in models.items():
        col = {}
        for v in LABEL:
            if v in r.params.index:
                col[LABEL[v]] = f"{r.params[v]:.3f}{stars(r.pvalues[v])}"
                col[LABEL[v] + " "] = f"({r.std_errors[v]:.3f})"
        if extra and name in extra:
            col.update(extra[name])
        col["Observations"] = str(int(r.nobs)); col["States"] = str(int(r.entity_info["total"]))
        T[name] = col
    rows = [k for v in LABEL.values() for k in (v, v + " ")]
    rows += sorted({k for c in T.values() for k in c if k not in rows and k not in ("Observations", "States")})
    rows += ["Observations", "States"]
    t = pd.DataFrame(T).reindex(rows).dropna(how="all").fillna("")
    t.index = [i if not i.endswith(" ") else "" for i in t.index]
    return t

# =====================================================================
# 7A. Manufacturing output: which credit matters?
# =====================================================================
Y = "ln_nsdp_manuf_cnst"
need = [Y, X, "l1_ln_small", "l1_ln_large", "l1_ln_grass", "l1_small_share", "l1_grass_share"] + CTRL
A = df.dropna(subset=need)                                   # same rows in every column
mA = {
    "(1) Small-ticket credit":   fe(Y, ["l1_ln_small"], A),
    "(2) Small vs large":        fe(Y, ["l1_ln_small", "l1_ln_large"], A),
    "(3) Grassroots credit":     fe(Y, ["l1_ln_grass"], A),
    "(4) Total + small share":   fe(Y, [X, "l1_small_share"], A),
    "(5) Total + grass share":   fe(Y, [X, "l1_grass_share"], A),
    "(6) Total x small share":   fe(Y, [X, "cr_x_small_share"], A),
}
b_d, se_d, p_d = lincom(mA["(2) Small vs large"], {"l1_ln_small": 1, "l1_ln_large": -1})
extra = {"(2) Small vs large": {"Small minus large": f"{b_d:.3f}{stars(p_d)}", "Small minus large ": f"({se_d:.3f})"}}
tA = make_table(mA, extra)
print("Table 7A. Which manufacturing credit is linked to manufacturing output?")
print("Dep. var: ln real mfg output. State + year FE, ln income pc (t-1). SE clustered by state.")
print("Small-ticket = loan cells averaging <= Rs 25 lakh per account. *** p<0.01, ** p<0.05, * p<0.10")
display(tA)
tA.to_csv("tables/table7a_channel.csv")

# =====================================================================
# 7B. Informal (unregistered) manufacturing output, FY2010-11 to FY2014-15 only
# =====================================================================
YU = "ln_nsdp_manuf_unreg_cnst"
B = df.dropna(subset=[YU, X, "l1_ln_small", "l1_ln_grass"] + CTRL)
mB = {
    "(1) Total credit":        fe(YU, [X], B),
    "(2) Small-ticket credit": fe(YU, ["l1_ln_small"], B),
    "(3) Grassroots credit":   fe(YU, ["l1_ln_grass"], B),
}
tB = make_table(mB)
print(f"\nTable 7B. Informal manufacturing output ({B['year'].min()-1}-{str(B['year'].min())[-2:]} "
      f"to {B['year'].max()-1}-{str(B['year'].max())[-2:]}; short panel - supporting evidence only)")
display(tB)
tB.to_csv("tables/table7b_informal.csv")

# =====================================================================
# 7C. Reliable p-values (wild cluster bootstrap) for the key coefficients
# =====================================================================
keyp = pd.DataFrame([
    ["7A(1) small-ticket credit", mA["(1) Small-ticket credit"].params["l1_ln_small"],
     mA["(1) Small-ticket credit"].pvalues["l1_ln_small"], wild_p(A, Y, "l1_ln_small", CTRL)],
    ["7A(3) grassroots credit", mA["(3) Grassroots credit"].params["l1_ln_grass"],
     mA["(3) Grassroots credit"].pvalues["l1_ln_grass"], wild_p(A, Y, "l1_ln_grass", CTRL)],
    ["7A(4) small-ticket share", mA["(4) Total + small share"].params["l1_small_share"],
     mA["(4) Total + small share"].pvalues["l1_small_share"], wild_p(A, Y, "l1_small_share", [X] + CTRL)],
    ["7A(6) total x small share", mA["(6) Total x small share"].params["cr_x_small_share"],
     mA["(6) Total x small share"].pvalues["cr_x_small_share"], wild_p(A, Y, "cr_x_small_share", [X] + CTRL)],
    ["7B(2) small-ticket -> informal", mB["(2) Small-ticket credit"].params["l1_ln_small"],
     mB["(2) Small-ticket credit"].pvalues["l1_ln_small"], wild_p(B, YU, "l1_ln_small", CTRL)],
], columns=["Coefficient", "estimate", "p (clustered)", "p (wild bootstrap)"])
print("\n7C. Key coefficients with wild-bootstrap p-values (trust these with ~30 states)")
display(keyp.round(4))
keyp.round(4).to_csv("tables/table7c_wild_p.csv", index=False)

# =====================================================================
# 7D. Plain-English reading
# =====================================================================
r1, r4 = mA["(1) Small-ticket credit"], mA["(4) Total + small share"]
print("\nWHAT THIS MEANS")
print(f"  - Q1: 10% more small-ticket credit last year -> {r1.params['l1_ln_small']*10:.2f}% output change "
      f"(wild p = {keyp.loc[0, 'p (wild bootstrap)']:.3f}).")
print(f"  - Q1: small minus large credit effect = {b_d:.3f} (p = {p_d:.3f}). "
      "Positive & significant = small loans do more per rupee than large loans.")
print(f"  - Q2: holding total credit fixed, raising the small-ticket share by 10 points "
      f"(e.g. 30% -> 40%) -> {r4.params['l1_small_share']*0.10*100:.2f}% output change "
      f"(wild p = {keyp.loc[2, 'p (wild bootstrap)']:.3f}).")
print(f"  - Q3: 10% more small-ticket credit -> {mB['(2) Small-ticket credit'].params['l1_ln_small']*10:.2f}% "
      f"informal output change (wild p = {keyp.loc[4, 'p (wild bootstrap)']:.3f}; only "
      f"{int(mB['(2) Small-ticket credit'].nobs)} rows, so low power).")
sig = keyp[(keyp["estimate"] > 0) & (keyp["p (wild bootstrap)"] < 0.10)]
print("  - Verdict: " + ("small-borrower credit shows a positive link in: " + ", ".join(sig["Coefficient"])
                         if len(sig) else "no reliable evidence that small-borrower credit raises output either."))

In [ ]:
# =====================================================================
# STEP 8: Reverse causality - does credit drive output, or output drive credit?
#   8A. Panel Granger regressions (two-way FE, growth rates)
#   8B. Dumitrescu-Hurlin panel Granger test (state-by-state)
#   8C. System GMM (Blundell-Bond), two-step, collapsed instruments, Windmeijer SEs
# Reads analysis_panel.csv -> tables/table8a..8c
# =====================================================================
import sys, subprocess
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from IPython.display import display
try:
    from linearmodels.panel import PanelOLS
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "linearmodels"])
    from linearmodels.panel import PanelOLS

Path("tables").mkdir(exist_ok=True)
df = pd.read_csv("analysis_panel.csv").sort_values(["state", "year"]).reset_index(drop=True)
GY = "g_nsdp_manuf_cnst_w" if "g_nsdp_manuf_cnst_w" in df else "g_nsdp_manuf_cnst"      # output growth
GX = "g_real_cr_out_manuf_w" if "g_real_cr_out_manuf_w" in df else "g_real_cr_out_manuf"  # credit growth

def lagk(data, col, k):
    g = data.groupby("state")
    return g[col].shift(k).where(data["year"] - g["year"].shift(k) == k)

for k in [1, 2]:
    df[f"L{k}_gy"] = lagk(df, GY, k)
    df[f"L{k}_gx"] = lagk(df, GX, k)

def pstars(p):
    return "***" if p < .01 else "**" if p < .05 else "*" if p < .10 else ""

# =====================================================================
# 8A. Panel Granger regressions with state + year FE (growth rates)
# =====================================================================
def granger_fe(dep, own, other, p):
    xs = [f"L{k}_{own}" for k in range(1, p + 1)] + [f"L{k}_{other}" for k in range(1, p + 1)]
    d = df.dropna(subset=[dep] + xs).set_index(["state", "year"])
    r = PanelOLS(d[dep], d[xs], entity_effects=True, time_effects=True).fit(cov_type="clustered", cluster_entity=True)
    test = [f"L{k}_{other}" for k in range(1, p + 1)]
    w = r.wald_test(formula=" = ".join([" + ".join(test), "0"]) if p == 1 else
                    ", ".join([f"{t} = 0" for t in test]))
    return {"sum of lag coefs": r.params[test].sum(), "Wald stat": float(w.stat), "p-value": float(w.pval),
            "N": int(r.nobs)}

rows = []
for p in [1, 2]:
    rows.append({"Direction": "Credit -> Output", "lags": p, **granger_fe(GY, "gy", "gx", p)})
    rows.append({"Direction": "Output -> Credit", "lags": p, **granger_fe(GX, "gx", "gy", p)})
t8a = pd.DataFrame(rows)
t8a["sig"] = t8a["p-value"].map(pstars)
print("8A. Panel Granger regressions (growth rates, state + year FE, SE clustered by state)")
print("    'sum of lag coefs' = combined effect of the other variable's past growth")
display(t8a.round(4))
t8a.round(4).to_csv("tables/table8a_granger_fe.csv", index=False)

# =====================================================================
# 8B. Dumitrescu-Hurlin (2012) test: one regression per state, then averaged
#     H0: no state has Granger causality; H1: at least some states do
# =====================================================================
def dumitrescu_hurlin(dep, own, other, p=1):
    W, Ts = [], []
    for st, s in df.groupby("state"):
        xs = [f"L{k}_{own}" for k in range(1, p + 1)] + [f"L{k}_{other}" for k in range(1, p + 1)]
        s = s.dropna(subset=[dep] + xs)
        T = len(s)
        if T < 2 * p + 6:                         # too short for the small-sample formula
            continue
        Xm = np.column_stack([np.ones(T), s[xs].values]); yv = s[dep].values
        b, *_ = np.linalg.lstsq(Xm, yv, rcond=None)
        e = yv - Xm @ b
        V = (e @ e) / (T - Xm.shape[1]) * np.linalg.pinv(Xm.T @ Xm)
        idx = list(range(1 + p, 1 + 2 * p))
        W.append(float(b[idx] @ np.linalg.pinv(V[np.ix_(idx, idx)]) @ b[idx])); Ts.append(T + p)
    W, Ts = np.array(W), np.array(Ts); N = len(W)
    EW = p * (Ts - 2 * p - 1) / (Ts - 2 * p - 3)
    VW = 2 * p * (Ts - 2 * p - 1) ** 2 * (Ts - p - 3) / ((Ts - 2 * p - 3) ** 2 * (Ts - 2 * p - 5))
    Zbar = np.sqrt(N / (2 * p)) * (W.mean() - p)
    Ztil = np.sqrt(N) * (W.mean() - EW.mean()) / np.sqrt(VW.mean())
    share = np.mean(W > stats.chi2.ppf(0.95, p))
    return {"states": N, "W-bar": W.mean(), "Z-bar": Zbar, "p (Z-bar)": 2 * (1 - stats.norm.cdf(abs(Zbar))),
            "Z-tilde (small T)": Ztil, "p (Z-tilde)": 2 * (1 - stats.norm.cdf(abs(Ztil))),
            "share of states significant at 5%": share}

t8b = pd.DataFrame([{"Direction": "Credit -> Output", **dumitrescu_hurlin(GY, "gy", "gx")},
                    {"Direction": "Output -> Credit", **dumitrescu_hurlin(GX, "gx", "gy")}])
print("\n8B. Dumitrescu-Hurlin panel Granger test (growth rates, 1 lag). Use Z-tilde: T is small.")
display(t8b.round(4))
t8b.round(4).to_csv("tables/table8b_dumitrescu_hurlin.csv", index=False)

# =====================================================================
# 8C. System GMM (Blundell-Bond 1998), two-step with Windmeijer (2005) correction
#     ln y_it = rho ln y_i,t-1 + beta ln credit_i,t-1 + gamma ln income_i,t-1 + year FE + state effect
#     credit and income treated as ENDOGENOUS; instruments collapsed (Roodman 2009)
# =====================================================================
def system_gmm(data, y, xvars, lags=(2, 3)):
    """y: outcome; xvars: other variables entering at lag 1 (endogenous). Returns dict of results."""
    d = data[["state", "year", y] + xvars].copy()
    states = sorted(d["state"].unique()); years = sorted(d["year"].unique())
    full = pd.MultiIndex.from_product([states, years], names=["state", "year"])
    d = d.set_index(["state", "year"]).reindex(full)
    lvl = {v: d[v].unstack().values for v in [y] + xvars}              # N x T arrays (NaN = missing)
    N, T = lvl[y].shape
    def L(a, k):                                                       # lag along time axis
        out = np.full_like(a, np.nan); out[:, k:] = a[:, :T - k]; return out
    reg = {"L1." + v: L(lvl[v], 1) for v in [y] + xvars}               # regressors at lag 1
    names = list(reg)
    yd = np.array(years)
    yeardum = [(yd == yr).astype(float) for yr in years[2:]]           # first 2 years lost to lags
    K = len(names) + len(yeardum) + 1                                  # + constant

    # GMM-style instrument columns (collapsed): for each variable, one column per lag
    gmm_vars = [y] + xvars
    a, b = lags
    rowsX, rowsY, rowsZ, unit = [], [], [], []
    nZd = len(gmm_vars) * (b - a + 1)                                  # diff-eq instruments
    nZl = len(gmm_vars)                                                # level-eq instruments
    nZiv = len(yeardum) + 1                                            # year dummies + const (level eq)
    Ltot = nZd + nZl + nZiv
    for i in range(N):
        for t in range(2, T):
            # ---- differenced equation (t needs t-1 and t-2 data) ----
            yv = lvl[y][i, t] - lvl[y][i, t - 1]
            xv = [reg[n][i, t] - reg[n][i, t - 1] for n in names]
            if np.isfinite(yv) and np.all(np.isfinite(xv)):
                z = np.zeros(Ltot); c = 0
                for v in gmm_vars:
                    for k in range(a, b + 1):
                        val = lvl[v][i, t - k] if t - k >= 0 else np.nan
                        z[c] = val if np.isfinite(val) else 0.0; c += 1
                xrow = xv + [yeardum[j][t] - yeardum[j][t - 1] for j in range(len(yeardum))] + [0.0]
                rowsX.append(xrow); rowsY.append(yv); rowsZ.append(z); unit.append((i, 0, t))
            # ---- level equation ----
            yl = lvl[y][i, t]
            xl = [reg[n][i, t] for n in names]
            if np.isfinite(yl) and np.all(np.isfinite(xl)):
                z = np.zeros(Ltot); c = nZd
                for v in gmm_vars:                                     # lagged difference, lag a-1
                    val = lvl[v][i, t - a + 1] - lvl[v][i, t - a] if t - a >= 0 else np.nan
                    z[c] = val if np.isfinite(val) else 0.0; c += 1
                z[nZd + nZl: nZd + nZl + len(yeardum)] = [yeardum[j][t] for j in range(len(yeardum))]
                z[-1] = 1.0
                xrow = xl + [yeardum[j][t] for j in range(len(yeardum))] + [1.0]
                rowsX.append(xrow); rowsY.append(yl); rowsZ.append(z); unit.append((i, 1, t))
    X = np.array(rowsX); Yv = np.array(rowsY); Z = np.array(rowsZ); U = np.array(unit)
    keep = Z.any(axis=0); Z = Z[:, keep]                               # drop all-zero instrument columns
    groups = [np.where(U[:, 0] == i)[0] for i in range(N)]
    groups = [g for g in groups if len(g)]

    def H_block(ix):                                                   # one-step weight: AB for diff, I for levels
        n = len(ix); H = np.eye(n); eq, tt = U[ix, 1], U[ix, 2]
        for p_ in range(n):
            for q in range(n):
                if eq[p_] == 0 and eq[q] == 0:
                    H[p_, q] = 2.0 if p_ == q else (-1.0 if abs(tt[p_] - tt[q]) == 1 else 0.0)
        return H
    A = Z.T @ X; zy = Z.T @ Yv
    W1 = np.linalg.pinv(sum(Z[g].T @ H_block(g) @ Z[g] for g in groups))
    b1 = np.linalg.pinv(A.T @ W1 @ A) @ A.T @ W1 @ zy
    e1 = Yv - X @ b1
    Om = sum(np.outer(Z[g].T @ e1[g], Z[g].T @ e1[g]) for g in groups)
    W2 = np.linalg.pinv(Om)
    V2 = np.linalg.pinv(A.T @ W2 @ A)
    b2 = V2 @ A.T @ W2 @ zy
    e2 = Yv - X @ b2
    g2 = Z.T @ e2
    # Windmeijer correction
    V1r = np.linalg.pinv(A.T @ W1 @ A) @ A.T @ W1 @ Om @ W1 @ A @ np.linalg.pinv(A.T @ W1 @ A)
    D = np.zeros((X.shape[1], X.shape[1]))
    for k in range(X.shape[1]):
        dOm = -sum(Z[g].T @ (np.outer(X[g, k], e1[g]) + np.outer(e1[g], X[g, k])) @ Z[g] for g in groups)
        D[:, k] = -V2 @ A.T @ W2 @ dOm @ W2 @ g2
    Vc = V2 + D @ V2 + (D @ V2).T + D @ V1r @ D.T
    se = np.sqrt(np.diag(Vc))
    # Hansen J
    J = float(g2 @ W2 @ g2); dfJ = Z.shape[1] - X.shape[1]
    # Arellano-Bond AR(m) tests on differenced residuals
    def ar_test(m):
        num, s1 = 0.0, 0.0; wX = np.zeros(X.shape[1]); Zew = np.zeros(Z.shape[1])
        for g in groups:
            w = np.zeros(len(g)); diff = U[g, 1] == 0; tt = U[g, 2]
            for j in np.where(diff)[0]:
                lagpos = np.where(diff & (tt == tt[j] - m))[0]
                if len(lagpos): w[j] = e2[g][lagpos[0]]
            w = w * diff
            we = w @ e2[g]; num += we; s1 += we ** 2
            wX += w @ X[g]; Zew += Z[g].T @ e2[g] * we
        var = s1 - 2 * wX @ V2 @ A.T @ W2 @ Zew + wX @ Vc @ wX
        zst = num / np.sqrt(var) if var > 0 else np.nan
        return zst, 2 * (1 - stats.norm.cdf(abs(zst)))
    ar1, ar2 = ar_test(1), ar_test(2)
    allnames = names + [f"yr{yr}" for yr in years[2:]] + ["const"]
    return {"b": pd.Series(b2, allnames), "se": pd.Series(se, allnames), "V": pd.DataFrame(Vc, allnames, allnames),
            "J": J, "J_df": dfJ, "J_p": 1 - stats.chi2.cdf(J, dfJ) if dfJ > 0 else np.nan,
            "AR1": ar1, "AR2": ar2, "instruments": Z.shape[1], "groups": len(groups),
            "obs_levels": int((U[:, 1] == 1).sum())}

Y, CR, INC = "ln_nsdp_manuf_cnst", "ln_real_cr_out_manuf", "ln_pc_nsdp_cnst"
res = {}
res["(1) Lags 2-3 (main)"] = system_gmm(df, Y, [CR, INC], lags=(2, 3))
res["(2) Lags 2-4"]        = system_gmm(df, Y, [CR, INC], lags=(2, 4))
res["(3) Lags 3-4"]        = system_gmm(df, Y, [CR, INC], lags=(3, 4))

out = {}
for name, r in res.items():
    b, se = r["b"], r["se"]
    col = {}
    for v, lab in [("L1." + Y, "ln output (t-1)"), ("L1." + CR, "ln real mfg credit (t-1)"),
                   ("L1." + INC, "ln income pc (t-1)")]:
        p = 2 * (1 - stats.norm.cdf(abs(b[v] / se[v])))
        col[lab] = f"{b[v]:.3f}{pstars(p)}"; col[lab + " "] = f"({se[v]:.3f})"
    rho, beta = b["L1." + Y], b["L1." + CR]
    grad = np.array([beta / (1 - rho) ** 2, 1 / (1 - rho)])           # delta method for beta/(1-rho)
    Vsub = r["V"].loc[["L1." + Y, "L1." + CR], ["L1." + Y, "L1." + CR]].values
    lr, lr_se = beta / (1 - rho), float(np.sqrt(grad @ Vsub @ grad))
    col["Long-run credit effect"] = f"{lr:.3f}{pstars(2 * (1 - stats.norm.cdf(abs(lr / lr_se))))}"
    col["Long-run credit effect "] = f"({lr_se:.3f})"
    col["AR(1) p-value"] = f"{r['AR1'][1]:.3f}"
    col["AR(2) p-value"] = f"{r['AR2'][1]:.3f}"
    col["Hansen J p-value"] = f"{r['J_p']:.3f}"
    col["Instruments / states"] = f"{r['instruments']} / {r['groups']}"
    col["Observations (levels)"] = str(r["obs_levels"])
    out[name] = col
t8c = pd.DataFrame(out)
t8c.index = [i if not i.endswith(" ") else "" for i in t8c.index]
print("\n8C. System GMM: ln real mfg output on its own lag, lagged credit and lagged income")
print("    Two-step, Windmeijer-corrected SE in brackets; credit & income treated as endogenous;")
print("    instruments collapsed; year dummies included. *** p<0.01, ** p<0.05, * p<0.10")
display(t8c)
t8c.to_csv("tables/table8c_system_gmm.csv")

main = res["(1) Lags 2-3 (main)"]
print("\nHOW TO CHECK THE GMM IS VALID (column 1)")
print(f"  - AR(1) p = {main['AR1'][1]:.3f}  (should be SMALL, < 0.10 - expected by construction)")
print(f"  - AR(2) p = {main['AR2'][1]:.3f}  (should be LARGE, > 0.10 - otherwise instruments are invalid)")
print(f"  - Hansen p = {main['J_p']:.3f}  (should be > 0.10 but NOT close to 1.000 - near 1 means too many instruments)")
print(f"  - Instruments {main['instruments']} vs states {main['groups']}  (instruments should not exceed states)")

In [ ]:
# =====================================================================
# STEP 8b: Is the positive System-GMM credit effect trustworthy?
#   (1) Difference GMM: uses ONLY within-state changes (no level equation)
#   (2) Difference-in-Hansen: are the extra level-equation instruments valid?
#   (3) Bond (2002) bracket check: GMM rho should lie between FE rho and OLS rho
# Run in the SAME notebook right after Step 8 (uses df, system_gmm, pstars)
# =====================================================================
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display
from linearmodels.panel import PanelOLS

Y, CR, INC = "ln_nsdp_manuf_cnst", "ln_real_cr_out_manuf", "ln_pc_nsdp_cnst"

def diff_gmm(data, y, xvars, lags=(2, 3)):
    """Arellano-Bond difference GMM, two-step, collapsed, Windmeijer SE. Same instruments as the
    differenced part of system_gmm; year dummies (differenced) instrument themselves."""
    d = data[["state", "year", y] + xvars].copy()
    states = sorted(d["state"].unique()); years = sorted(d["year"].unique())
    d = d.set_index(["state", "year"]).reindex(pd.MultiIndex.from_product([states, years]))
    lvl = {v: d[v].unstack().values for v in [y] + xvars}
    N, T = lvl[y].shape
    lag1 = {v: np.c_[np.full((N, 1), np.nan), lvl[v][:, :-1]] for v in [y] + xvars}
    yd = np.array(years); yeardum = [(yd == yr).astype(float) for yr in years[3:]]  # drop one more for collinearity
    a, b = lags; rows = []
    for i in range(N):
        for t in range(2, T):
            dy = lvl[y][i, t] - lvl[y][i, t - 1]
            dx = [lag1[v][i, t] - lag1[v][i, t - 1] for v in [y] + xvars]
            if not (np.isfinite(dy) and np.all(np.isfinite(dx))):
                continue
            z = []
            for v in [y] + xvars:
                for k in range(a, b + 1):
                    val = lvl[v][i, t - k] if t - k >= 0 else np.nan
                    z.append(val if np.isfinite(val) else 0.0)
            ddum = [yeardum[j][t] - yeardum[j][t - 1] for j in range(len(yeardum))]
            rows.append((i, t, dy, dx + ddum, z + ddum))
    U = np.array([[r[0], r[1]] for r in rows]); Yv = np.array([r[2] for r in rows])
    X = np.array([r[3] for r in rows]); Z = np.array([r[4] for r in rows])
    Z = Z[:, Z.any(axis=0)]
    groups = [np.where(U[:, 0] == i)[0] for i in range(N)]; groups = [g for g in groups if len(g)]
    def H(ix):
        tt = U[ix, 1]; n = len(ix)
        return np.array([[2.0 if p == q else (-1.0 if abs(tt[p] - tt[q]) == 1 else 0.0) for q in range(n)] for p in range(n)])
    A = Z.T @ X; zy = Z.T @ Yv
    W1 = np.linalg.pinv(sum(Z[g].T @ H(g) @ Z[g] for g in groups))
    b1 = np.linalg.pinv(A.T @ W1 @ A) @ A.T @ W1 @ zy; e1 = Yv - X @ b1
    Om = sum(np.outer(Z[g].T @ e1[g], Z[g].T @ e1[g]) for g in groups); W2 = np.linalg.pinv(Om)
    V2 = np.linalg.pinv(A.T @ W2 @ A); b2 = V2 @ A.T @ W2 @ zy; e2 = Yv - X @ b2; g2 = Z.T @ e2
    V1r = np.linalg.pinv(A.T @ W1 @ A) @ A.T @ W1 @ Om @ W1 @ A @ np.linalg.pinv(A.T @ W1 @ A)
    D = np.zeros((X.shape[1], X.shape[1]))
    for k in range(X.shape[1]):
        dOm = -sum(Z[g].T @ (np.outer(X[g, k], e1[g]) + np.outer(e1[g], X[g, k])) @ Z[g] for g in groups)
        D[:, k] = -V2 @ A.T @ W2 @ dOm @ W2 @ g2
    Vc = V2 + D @ V2 + (D @ V2).T + D @ V1r @ D.T
    names = ["L1." + v for v in [y] + xvars]
    J = float(g2 @ W2 @ g2); dfJ = Z.shape[1] - X.shape[1]
    return {"b": pd.Series(b2[:len(names)], names), "se": pd.Series(np.sqrt(np.diag(Vc))[:len(names)], names),
            "J": J, "J_df": dfJ, "J_p": 1 - stats.chi2.cdf(J, dfJ), "instruments": Z.shape[1], "groups": len(groups)}

rows = []
for lags in [(2, 3), (2, 4)]:
    s = system_gmm(df, Y, [CR, INC], lags=lags)
    dg = diff_gmm(df, Y, [CR, INC], lags=lags)
    C = s["J"] - dg["J"]; Cdf = s["J_df"] - dg["J_df"]
    for name, r in [("System GMM", s), ("Difference GMM", dg)]:
        bcr, scr = r["b"]["L1." + CR], r["se"]["L1." + CR]
        rows.append({"Lags": f"{lags[0]}-{lags[1]}", "Estimator": name,
                     "rho (lagged output)": r["b"]["L1." + Y],
                     "credit coef": bcr, "credit se": scr,
                     "credit p": 2 * (1 - stats.norm.cdf(abs(bcr / scr))),
                     "Hansen p": r["J_p"], "Instruments": r["instruments"],
                     "Diff-in-Hansen p (level instr.)": (1 - stats.chi2.cdf(C, Cdf)) if (name == "System GMM" and Cdf > 0) else np.nan})
t = pd.DataFrame(rows)
t["sig"] = t["credit p"].map(pstars)
print("8b-1/2. System vs Difference GMM (credit & income endogenous, collapsed, Windmeijer SE)")
print("        Diff-in-Hansen p < 0.10 means the extra LEVEL instruments of System GMM look invalid.")
display(t.round(3))
t.round(4).to_csv("tables/table8d_gmm_checks.csv", index=False)

# ---------- (3) Bond bracket: OLS rho is biased up, FE rho biased down; GMM should sit between ----------
d = df.sort_values(["state", "year"]).copy()
g = d.groupby("state"); ok = d["year"] - g["year"].shift(1) == 1
for v in [Y, CR, INC]:
    d["L1_" + v] = g[v].shift(1).where(ok)
xs = ["L1_" + Y, "L1_" + CR, "L1_" + INC]
dd = d.dropna(subset=[Y] + xs).set_index(["state", "year"])
ols = PanelOLS(dd[Y], dd[xs].assign(const=1.0), time_effects=True).fit(cov_type="clustered", cluster_entity=True)
fe = PanelOLS(dd[Y], dd[xs], entity_effects=True, time_effects=True).fit(cov_type="clustered", cluster_entity=True)
s_main = system_gmm(df, Y, [CR, INC], lags=(2, 3))
br = pd.DataFrame({
    "Pooled OLS (rho biased UP)":   [ols.params["L1_" + Y], ols.params["L1_" + CR]],
    "System GMM (lags 2-3)":        [s_main["b"]["L1." + Y], s_main["b"]["L1." + CR]],
    "Fixed effects (rho biased DOWN)": [fe.params["L1_" + Y], fe.params["L1_" + CR]],
}, index=["rho (lagged output)", "credit coef"])
inside = br.loc["rho (lagged output)", "Fixed effects (rho biased DOWN)"] <= br.loc["rho (lagged output)", "System GMM (lags 2-3)"] \
         <= br.loc["rho (lagged output)", "Pooled OLS (rho biased UP)"]
print("\n8b-3. Bond (2002) bracket check (dynamic models with year effects)")
display(br.round(3))
print(f"  GMM rho inside the [FE, OLS] range: {'YES' if inside else 'NO - a warning sign for the GMM instruments'}")

In [ ]:
# =====================================================================
# STEP 9: Robustness - rerun the three main models under four changes
#   Models: H1 (credit effect), H2 (credit x initial depth; T1 - T3), small-ticket channel
#   Changes: drop Maharashtra & Delhi | growth rates | total credit | drop COVID years
# All: state + year FE, ln income pc (t-1), SE clustered by state + wild-bootstrap p
# Reads analysis_panel.csv -> tables/table9_robustness.csv
# =====================================================================
import sys, subprocess
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from IPython.display import display
try:
    from linearmodels.panel import PanelOLS
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "linearmodels"])
    from linearmodels.panel import PanelOLS

Path("tables").mkdir(exist_ok=True)
df = pd.read_csv("analysis_panel.csv").sort_values(["state", "year"]).reset_index(drop=True)
g = df.groupby("state")
consec = df["year"] - g["year"].shift(1) == 1
def lag1(s):
    return s.groupby(df["state"]).shift(1).where(consec)
def ln(s):
    return np.log(s.where(s > 0))

# ---------- variables needed by every version ----------
df["ln_small"] = ln(np.exp(df["ln_real_cr_out_manuf"]) * df["small_ticket_share_manuf"])
df["l1_ln_small"] = lag1(df["ln_small"])
df["l1_ln_tot"] = lag1(df["ln_real_cr_out_total"]) if "ln_real_cr_out_total" in df else np.nan
gx = "g_real_cr_out_manuf_w" if "g_real_cr_out_manuf_w" in df else "g_real_cr_out_manuf"
gy = "g_nsdp_manuf_cnst_w" if "g_nsdp_manuf_cnst_w" in df else "g_nsdp_manuf_cnst"
df["l1_g_cr"] = lag1(df[gx])
df["l1_g_small"] = lag1(df["ln_small"] - df["ln_small"].groupby(df["state"]).shift(1).where(consec))
CTRL = ["l1_ln_pc_nsdp_cnst"]

VERSIONS = [
    # name,                          rows to keep,                         outcome,               credit (t-1),               small-ticket (t-1)
    ("Baseline (Tables 5-7)",        lambda d: d,                          "ln_nsdp_manuf_cnst",  "l1_ln_real_cr_out_manuf",  "l1_ln_small"),
    ("Drop Maharashtra & Delhi",     lambda d: d[d["branch_hub"] == 0],    "ln_nsdp_manuf_cnst",  "l1_ln_real_cr_out_manuf",  "l1_ln_small"),
    ("Growth rates",                 lambda d: d,                          gy,                    "l1_g_cr",                  "l1_g_small"),
    ("Total credit (all sectors)",   lambda d: d,                          "ln_nsdp_manuf_cnst",  "l1_ln_tot",                None),
    ("Drop COVID years (FY21, FY22)", lambda d: d[~d["year"].isin([2021, 2022])], "ln_nsdp_manuf_cnst", "l1_ln_real_cr_out_manuf", "l1_ln_small"),
]

def fe(y, xs, d):
    d = d.dropna(subset=[y] + xs + CTRL).set_index(["state", "year"])
    return PanelOLS(d[y], d[xs + CTRL].assign(const=1.0), entity_effects=True, time_effects=True,
                    drop_absorbed=True).fit(cov_type="clustered", cluster_entity=True)

def lincom(r, w):
    v = np.array([w.get(k, 0.0) for k in r.params.index])
    b = float(v @ r.params.values); se = float(np.sqrt(v @ r.cov.values @ v))
    return b, 2 * (1 - stats.t.cdf(abs(b / se), df=r.entity_info["total"] - 1))

def wild_p(d, y, x, others, reps=1999, seed=42):
    d = d.dropna(subset=[y, x] + others).reset_index(drop=True)
    Z = pd.get_dummies(d[["state", "year"]].astype(str), drop_first=True, dtype=float)
    Z = np.column_stack([np.ones(len(d)), Z.values, d[others].values])
    M = np.eye(len(d)) - Z @ np.linalg.pinv(Z)
    xt, yt = M @ d[x].values, M @ d[y].values
    gid = pd.factorize(d["state"])[0]; G = gid.max() + 1; n, k = len(d), Z.shape[1] + 1
    def t(yy):
        b = xt @ yy / (xt @ xt); e = yy - b * xt
        s = np.bincount(gid, weights=xt * e, minlength=G)
        return b / np.sqrt((G / (G - 1)) * ((n - 1) / (n - k)) * (s @ s) / (xt @ xt) ** 2)
    t0 = t(yt); rng = np.random.default_rng(seed)
    return np.mean(np.abs([t(M @ (yt * rng.choice([-1.0, 1.0], G)[gid])) for _ in range(reps)]) >= abs(t0))

def stars(p):
    return "***" if p < .01 else "**" if p < .05 else "*" if p < .10 else ""
def fmt(b, p, pw=None):
    s = f"{b:.3f}{stars(p)} (p={p:.2f})"
    return s + (f" [wild {pw:.2f}]" if pw is not None else "")

rows, raw = [], []
for name, keep, Y, X, S in VERSIONS:
    d = keep(df).copy()
    if d[X].notna().sum() == 0:
        print(f"Skipping '{name}': {X} not available"); continue
    d = d.dropna(subset=[Y, X] + CTRL)
    # H1
    r1 = fe(Y, [X], d); b1, p1 = r1.params[X], r1.pvalues[X]; w1 = wild_p(d, Y, X, CTRL)
    # H2: interaction with centred initial depth, and slope per tercile
    dep = d.groupby("state")["initial_depth"].first()
    d["cx_depth"] = d[X] * (d["initial_depth"] - dep.mean())
    for t in [1, 2, 3]:
        d[f"cx_T{t}"] = d[X] * (d["depth_tercile"] == t)
    r2 = fe(Y, [X, "cx_depth"], d); b2, p2 = r2.params["cx_depth"], r2.pvalues["cx_depth"]
    w2 = wild_p(d, Y, "cx_depth", [X] + CTRL)
    r3 = fe(Y, ["cx_T1", "cx_T2", "cx_T3"], d); b3, p3 = lincom(r3, {"cx_T1": 1, "cx_T3": -1})
    # Small-ticket channel
    if S is not None and d[S].notna().any():
        r4 = fe(Y, [S], d); b4, p4 = r4.params[S], r4.pvalues[S]; w4 = wild_p(d, Y, S, CTRL)
        small = fmt(b4, p4, w4)
    else:
        b4 = p4 = w4 = np.nan; small = "n/a (only defined for mfg credit)"
    rows.append({"Version": name, "States": int(r1.entity_info["total"]), "N": int(r1.nobs),
                 "H1: credit effect": fmt(b1, p1, w1),
                 "H2: credit x initial depth (H2 wants < 0)": fmt(b2, p2, w2),
                 "H2: T1 minus T3 (H2 wants > 0)": fmt(b3, p3),
                 "Small-ticket credit effect": small})
    raw.append({"Version": name, "H1_b": b1, "H1_p_wild": w1, "H2_int_b": b2, "H2_int_p_wild": w2,
                "H2_T1mT3_b": b3, "H2_T1mT3_p": p3, "Small_b": b4, "Small_p_wild": w4})

t9 = pd.DataFrame(rows).set_index("Version")
print("Table 9. Robustness of the main results")
print("Each cell: coefficient, stars from clustered SE, (clustered p) [wild-bootstrap p].")
print("Growth-rate version: output growth on last year's credit growth.")
display(t9)
t9.to_csv("tables/table9_robustness.csv")
pd.DataFrame(raw).round(4).to_csv("tables/table9_robustness_raw.csv", index=False)

# ---------- one-line verdicts ----------
raw = pd.DataFrame(raw)
print("\nVERDICTS ACROSS ALL VERSIONS (using wild-bootstrap p < 0.10)")
h1 = ((raw["H1_b"] > 0) & (raw["H1_p_wild"] < .10)).sum()
h2 = ((raw["H2_int_b"] < 0) & (raw["H2_int_p_wild"] < .10)).sum()
sm = ((raw["Small_b"] > 0) & (raw["Small_p_wild"] < .10)).sum()
n, ns = len(raw), raw["Small_b"].notna().sum()
print(f"  - H1 positive & significant in {h1} of {n} versions")
print(f"  - H2 (negative interaction) significant in {h2} of {n} versions")
print(f"  - Small-ticket credit positive & significant in {sm} of {ns} versions")